# Research Ownership Lab — LVBench → Qwen3-VL → VidCom2 → answer

**Purpose.** Take *one real LVBench question* and trace it, with measured numbers, through
every stage from the dataset row to the scored answer. By the end you should be able to
stand at a whiteboard and explain this system without a coding agent in the room.

This is **not** a diagnostic notebook. Every code cell is preceded by an explanation of
what we are about to learn, where we are in the pipeline, what is going in, and what you
should predict — and followed by an interpretation of what actually came out.

---

## How to read this notebook

Run **one cell at a time**. Before each measurement cell there is a
**"Before you run this"** box. Write your prediction down (on paper, in a comment, out
loud — it does not matter) *before* executing. The point is to practise reasoning about
the system, not to read outputs.

## The four kinds of claim

One of the ways ownership of a project gets lost is that *config values*, *paper claims*,
and *runtime behaviour* become mentally interchangeable. Every substantive statement in
this notebook is tagged as exactly one of:

| tag | means |
|---|---|
| 🟩 **RUNTIME** | We executed something and printed the number. It is a fact about this box, this checkpoint, this video. |
| 🟦 **SOURCE** | Read off the actual code in this repo or in the installed vLLM/transformers. Verifiable by opening the file at the line given. |
| 🟨 **PAPER** | What a paper claims. May or may not describe the code that ships. |
| 🟧 **OURS** | Our interpretation, hypothesis, or experiment-specific choice. Not established fact. |

If you catch yourself repeating a 🟨 or 🟧 claim as if it were 🟩, that is the failure
mode this notebook exists to prevent.

## What this notebook will show you that you probably do not expect

Three findings surfaced while building it. They are stated here so you can look for them,
and each is verified in place:

1. 🟩 **`--mm-processor-kwargs '{"max_pixels": ...}'` has no effect on the video grid in
   vLLM 0.19.** Every video arm in this project has been running at the checkpoint's
   default pixel budget, not at LongVT's 224² and not at arm e/f's 786432. (§5)
2. 🟩 **Qwen3-VL's video `max_pixels` is a *whole-video* budget, not a per-frame one.**
   Asking for more frames therefore *lowers* per-frame resolution automatically. This is
   why the "matched budget" between run1 and run3 is not actually matched. (§5)
3. 🟩 **VidCom2's adaptive per-frame budget is effectively degenerate at our operating
   points** — 124 of 128 frames receive an identical budget. Its softmax temperature of
   0.01 collapses the allocation to near-one-hot. (§7)

---
# §0 · Setup

## What are we trying to learn from this cell?
Which interpreter, which vLLM, which transformers, which checkpoint. Nothing downstream
means anything if this is wrong.

## Why it matters for the experiment
🟦 **SOURCE** — `RESEARCH.md` records that `conda run -n vllm` on this box silently
resolves to a *different* env (`dvd_tool`, python 3.10, vLLM 0.16.0). Several older notes
in that file were written under that confusion and say "0.16" when the real serving env is
0.19.0. Two envs means two different `compute_retention_mask` implementations, two
different processors, and results that cannot be compared.

## Where are we in the pipeline?
Nowhere yet — this is the ground under the pipeline.

## What should you expect?
`python 3.12.x`, `vllm 0.19.0`, an interpreter path containing `envs/vllm/`.

## What would indicate something is wrong?
Any python 3.10, any vLLM that is not 0.19.0, or an interpreter outside `envs/vllm`.

In [1]:
import os, sys, json, math, subprocess, textwrap
sys.path.insert(0, "/home/cfyang/hanklin")

import numpy as np
import torch

import vllm, transformers
print("interpreter  :", sys.executable)
print("python       :", sys.version.split()[0])
print("vllm         :", vllm.__version__)
print("transformers :", transformers.__version__)
print("torch        :", torch.__version__)

MODEL = ("/local1/cfyang/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/"
         "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b")
OUT   = "/local1/cfyang/hanklin/outputs/lvbench_agent"
PROXY_DIR  = f"{OUT}/skim_proxies"          # 448-wide proxies  (run1-run4)
PROXY_HIRES = f"{OUT}/skim_proxies_hires"   # 1280-wide proxies (arm e/f)

assert sys.version_info[:2] == (3, 12), "wrong conda env -- see the note above"
assert vllm.__version__ == "0.19.0", f"expected vLLM 0.19.0, got {vllm.__version__}"
assert "envs/vllm/" in sys.executable
print("\nenvironment OK")

/local1/cfyang/miniconda3/envs/vllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


interpreter  : /local1/cfyang/miniconda3/envs/vllm/bin/python
python       : 3.12.13
vllm         : 0.19.0
transformers : 4.57.6
torch        : 2.10.0+cu129

environment OK


### Interpreting the output

🟩 **RUNTIME** — you should have seen `3.12.13 / vllm 0.19.0 / transformers 4.57.6 /
torch 2.10.0+cu129`.

Two things follow that matter later:

- vLLM **0.19.0** ships EVS only. 🟦 **SOURCE**: `vllm/multimodal/evs.py` defines
  `compute_retention_mask` (EVS) and `compute_retained_tokens_count`, and there is no
  `VideoPruningMethod` enum and no `vidcom2` anywhere in the wheel. VidCom2 reaches this
  server only through the out-of-tree plugin in `longvt_compression/vidcom2_vllm/`. §7
  traces exactly how.
- The vision-side geometry constants come from the **checkpoint**, not from vLLM. We read
  them next.

### 🔧 Exercise 0
Run `grep -rn "vidcom2" /local1/cfyang/miniconda3/envs/vllm/lib/python3.12/site-packages/vllm/ | wc -l`
yourself. Convince yourself the answer is 0, and that therefore *nothing in vLLM knows
what VidCom2 is*. Everything that makes it work is in this repo.

## §0.2 · The geometry constants

## What are we trying to learn from this cell?
The five numbers that turn "a video" into "a token count":
`patch_size`, `temporal_patch_size`, `spatial_merge_size`, `out_hidden_size`, and the
video processor's `size` budget.

## Why it matters
Every token-budget calculation in this project is these five numbers plus arithmetic. If
you can hold them in your head you can predict a token count without running anything —
which is the whole point of this notebook.

## What is entering this step?
`config.json` and `preprocessor_config.json` from the checkpoint directory.

## Before you run this — predict
Qwen3-VL is a 16×16-patch ViT with a 2×2 spatial merger and a temporal patch size of 2.
Given that, **how many raw video frames produce one temporal grid step? How many ViT
patches collapse into one token the LLM sees?**

In [2]:
from transformers import AutoConfig, AutoProcessor

cfg  = AutoConfig.from_pretrained(MODEL)
proc = AutoProcessor.from_pretrained(MODEL)
vp   = proc.video_processor          # Qwen3VLVideoProcessor

VC = cfg.vision_config
print("vision_config")
for k in ("patch_size", "temporal_patch_size", "spatial_merge_size",
          "hidden_size", "out_hidden_size", "depth", "deepstack_visual_indexes"):
    print(f"  {k:24} = {getattr(VC, k)}")

print("\nvideo_processor (Qwen3VLVideoProcessor)")
for k in ("patch_size", "temporal_patch_size", "merge_size",
          "min_frames", "max_frames", "fps", "do_sample_frames", "do_resize", "size"):
    print(f"  {k:24} = {getattr(vp, k, '<none>')}")

print("\nspecial tokens")
for k in ("video_token_id", "image_token_id", "vision_start_token_id",
          "vision_end_token_id"):
    print(f"  {k:24} = {getattr(cfg, k)}")

PATCH, TPS, MERGE = VC.patch_size, VC.temporal_patch_size, VC.spatial_merge_size
VIDEO_TOKEN_ID = cfg.video_token_id

vision_config
  patch_size               = 16
  temporal_patch_size      = 2
  spatial_merge_size       = 2
  hidden_size              = 1152
  out_hidden_size          = 4096
  depth                    = 27
  deepstack_visual_indexes = [8, 16, 24]

video_processor (Qwen3VLVideoProcessor)
  patch_size               = 16
  temporal_patch_size      = 2
  merge_size               = 2
  min_frames               = 4
  max_frames               = 768
  fps                      = 2
  do_sample_frames         = True
  do_resize                = True
  size                     = {'longest_edge': 25165824, 'shortest_edge': 4096}

special tokens
  video_token_id           = 151656
  image_token_id           = 151655
  vision_start_token_id    = 151652
  vision_end_token_id      = 151653


### Interpreting the output

🟩 **RUNTIME** — `patch_size=16`, `temporal_patch_size=2`, `spatial_merge_size=2`,
`out_hidden_size=4096`, `size={'longest_edge': 25165824, 'shortest_edge': 4096}`.

Read those as one sentence:

> The ViT cuts each frame into 16×16 pixel patches. It consumes **2 raw frames at a time**
> as one temporal patch. After the encoder, a PatchMerger folds each **2×2 block of
> spatial patches** into one token of width **4096**, which is the LLM's hidden size.

So the arithmetic that governs everything is:

```text
grid_t          = ceil(raw_frames / temporal_patch_size)   = raw_frames / 2
grid_h, grid_w  = resized_px_h / 16 , resized_px_w / 16
tokens/grid-frame (tpf) = (grid_h / 2) * (grid_w / 2)
total visual tokens     = grid_t * tpf
```

**That `size` dict is the one to stare at.** 🟦 **SOURCE** —
`transformers/models/qwen3_vl/video_processing_qwen3_vl.py:34-64`, `smart_resize`:

```python
t_bar = round(num_frames / temporal_factor) * temporal_factor
if t_bar * h_bar * w_bar > max_pixels:                      # <-- NOTE: t_bar is IN HERE
    beta = math.sqrt((num_frames * height * width) / max_pixels)
    h_bar = max(factor, math.floor(height / beta / factor) * factor)
    w_bar = max(factor, math.floor(width  / beta / factor) * factor)
```

`max_pixels` (= `size["longest_edge"]`) is compared against `t_bar * h_bar * w_bar` — the
**whole video's pixel volume**, not one frame's. §5 shows the consequence, which is
large and counter-intuitive.

### 🔧 Exercise 0.2
Open `video_processing_qwen3_vl.py` and read `smart_resize` yourself. Then answer without
running anything: *if I double the number of frames and change nothing else, what happens
to the per-frame resolution?*

---
# §1 · The map

Before touching a single tensor, here is the whole path in one table. Keep this open in a
second window; the rest of the notebook walks it top to bottom.

```text
 LVBench row
  |
  v
 source MP4
  |
  v
 proxy MP4
  |
  v
 vLLM video loader
  |
  v
 Qwen video processor
  |
  v
 grid_thw
  |
  v
 ViT
  |
  v
 PatchMerger
  |
  v
 VidCom2 / EVS
  |
  v
 retention mask
  |
  v
 mRoPE
  |
  v
 LLM prefill
  |
  v
 generation
  |
  v
 tool loop
  |
  v
 answer parsing
  |
  v
 scoring
```

| # | stage | file · function | key config | in → out | what silently goes wrong |
|---|---|---|---|---|---|
| 1 | dataset row | `fast_agent/data.py::load_lvbench` | `LVBENCH_META` | jsonl → `dict` with `evidence` | `time_reference` malformed (2 rows) or zero-width (263 rows) → `evidence=None`, question silently unscored for localization |
| 2 | question text | `data.py::format_question`, `annotate_clock_times` | `FA_UNIFY_TIME_FORMAT` | str → str | clock times (`17:16`) read by the model as *seconds* |
| 3 | proxy build | `fast_agent/make_skim_proxy.py::build_one` | `--frames`, `--max-side` | 232 MB mp4 → 5 MB mp4 | proxy duration ≠ source duration → every `<t seconds>` marker is wrong, silently |
| 4 | request | `fast_agent/run_agent.py::_skim_content` | `--skim-mode`, `--proxy-dir`, `--require-proxy` | row → `{"type":"video_url","url":"file://…"}` | no `--require-proxy` → falls back to the 62 GB source, 39 s/request |
| 5 | video load | `vllm/multimodal/video.py::OpenCVVideoBackend.load_bytes` | `--media-io-kwargs num_frames` | mp4 bytes → `(N,H,W,3)` uint8 + metadata | flag missing → vLLM resamples to its own default; a 256-frame request costs the same as 64 |
| 6 | timestamps | `vllm/…/qwen3_vl.py::_calculate_timestamps` (l. 814) | `temporal_patch_size` | frame indices + fps → `grid_t` floats | proxy fps wrong → markers off by orders of magnitude |
| 7 | resize + patchify | `transformers/…/video_processing_qwen3_vl.py::smart_resize` | `size.longest_edge` | `(N,H,W,3)` → `pixel_values_videos`, `video_grid_thw` | `max_pixels` kwarg is **inert** for video (§5); budget is whole-video |
| 8 | placeholder sizing | `vllm/…/qwen3_vl.py::get_video_repl` (l. 1258) + `evs.py::compute_retained_tokens_count` | `--video-pruning-rate` | grid → `<\|video_pad\|>` run length | count here must equal the mask's `True` count *exactly* or generation corrupts |
| 9 | ViT | `Qwen3VLVisionModel.forward` | 27 layers, hidden 1152 | `(n_patch, 1536)` → `(n_tok, 4096)` | none commonly — but note pruning happens **after** this, so it saves no ViT compute |
| 10 | PatchMerger | inside the vision model | `spatial_merge_size=2` | 4 patches → 1 token | — |
| 11 | pruning | `vllm/…/qwen3_vl.py::_postprocess_video_embeds_evs` (l. 1891) → `compute_retention_mask` | `--video-pruning-rate`, `FA_PRUNE_METHOD` | `(T·tpf, 4096)` → `(kept, 4096)` | patch not applied in EngineCore → stock EVS runs while you believe VidCom2 did |
| 12 | mRoPE | `evs.py::compute_mrope_for_media`, `recompute_mrope_positions`; `qwen3_vl.py::_get_expanded_positions` (l. 2049) | — | positions recomputed from the **unpruned** grid, then indexed by the mask | — |
| 13 | prefill | LLM | `--max-model-len 40960` | `(seq, 4096)` | overflow on wide skim + 5 crops → `BadRequestError` |
| 14 | tool loop | `run_agent.py::answer_one` | `--tools`, `--max-rounds`, `--prompt-style` | messages → messages | no-tool arm handed the tool prompt (the 2026-08-13 run1 bug) |
| 15 | crop tool | `fast_agent/tools.py::crop_frames` | `CROP_MAX_FRAMES=128`, `FPS=1`, `MAX_PIXELS=224²` | span → ≤128 PIL images | reads the **real** file, so it obeys a wrong timestamp silently |
| 16 | parse | `data.py::extract_answer` | — | text → `"A".."D"` or `None` | refusal enumerations scored as a confident `D` |
| 17 | score | `run_agent.py::_span_metrics`, `write_summary` | `IOU_MIN_WIDTH=16` | calls + GT → `covered_frac`, `landed` | `error` rows written as `pred=None, correct=False` and quoted as accuracy |

### After this section, I should be able to explain
- the sixteen stages in order, and which file owns each one
- which three stages can be wrong *without raising an exception* (3, 5, 11)
- why stage 8 and stage 11 must agree to the token

---
# §2 · Stage 1–2 · The dataset row

```text
 MP4   <-- WE ARE HERE
  v
 Proxy
  v
 Loader
  v
 Processor
  v
 ViT
  v
 Merger
  v
 VidCom2
  v
 mRoPE
  v
 LLM
  v
 Answer
```

## What are we trying to learn from this cell?
What one LVBench row actually contains, and in particular what `evidence` is — the
ground-truth time span that makes this project's headline metric measurable at all.

## Why it matters for the experiment
🟧 **OURS** — the entire ACTIVE TASK ("does compression improve temporal localization?")
rests on LVBench shipping a per-question evidence span. VideoMME does not have one; that
is *the* reason this is an LVBench task.

## Where are we?
Stage 1. Nothing has been decoded yet.

## What is entering this step?
`video_info.meta.jsonl` from the zai-org LVBench snapshot.
🟦 **SOURCE** — `data.py:180-221`. Note it reads the **jsonl**, not `LVBench.tsv`: the tsv
drops `time_reference`, which is the field we need.

## Before you run this — predict
LVBench has 1,549 questions over 103 videos. **Roughly how many questions per video?** And
if a video is ~1 hour long and an evidence span is ~15 s, **what fraction of the video is
the answer actually in?**

In [3]:
from longvt_compression.fast_agent import config, data, make_skim_proxy, tools

rows = data.load_dataset("lvbench", n=None, seed=0)
print(f"rows          : {len(rows)}")
print(f"videos        : {len(set(r['videoID'] for r in rows))}")
print(f"q per video   : {len(rows)/len(set(r['videoID'] for r in rows)):.1f}")

ev  = [r for r in rows if r["evidence"]]
zw  = [r for r in ev if r["evidence"][1] <= r["evidence"][0]]
triv = [r for r in rows if r["localization_trivial"]]
print(f"with evidence : {len(ev)}   malformed: {len(rows)-len(ev)}   zero-width: {len(zw)}")
print(f"localization-TRIVIAL (question states its own timestamp): {len(triv)}"
      f"  = {100*len(triv)/len(rows):.1f}%")

w = np.array([r["evidence"][1]-r["evidence"][0] for r in ev if r["evidence"][1] > r["evidence"][0]])
print(f"\nevidence-span width (s): p25={np.percentile(w,25):.0f} median={np.median(w):.0f} "
      f"p75={np.percentile(w,75):.0f} max={w.max():.0f}")

rows          : 1549
videos        : 103
q per video   : 15.0
with evidence : 1547   malformed: 2   zero-width: 263
localization-TRIVIAL (question states its own timestamp): 204  = 13.2%

evidence-span width (s): p25=9 median=27 p75=112 max=7865


### Interpreting the output

🟩 **RUNTIME** — 1,549 questions / 103 videos ≈ 15 questions per video. 1,547 have a
parseable evidence span; 263 of those are **zero-width** (`19:30-19:30`), and 2 are
malformed.

Two consequences you should carry forward:

1. **One video serves ~15 questions.** That is why `--mm-processor-cache-gb 0` matters
   for server stability, and why building one proxy per video pays for itself 15×.
2. 🟦 **SOURCE** — `data.py:209-213` flags `localization_trivial`: 13.2% of questions
   *state their own timestamp* in the stem ("What happens from 17:16-17:40?"). Those
   questions need no search at all. 🟧 **OURS** — any hit-rate number that does not
   stratify on this flag is inflated by that 13%.

### 🔧 Exercise 2
Open `data.py` and find the line that sets `localization_trivial`. Explain in one sentence
what `_CLOCK_RE` matches and why a question containing `17:16` is unfair to include in a
localization score.

## §2.2 · Choose the one sample we will follow

## What are we trying to learn?
We fix a single question now and carry it to the end of the notebook. Everything numeric
from here on refers to *this* row.

## Why it matters
🟧 **OURS** — aggregate accuracy tells you nothing about mechanism. A single traced
example tells you what the model actually saw and why it answered what it answered.

## Selection criteria
A real (non-zero-width) evidence span, and all three proxies already built (64-frame,
256-frame, 32-frame hi-res) so we can compare arms on identical content.

## Before you run this — predict
Given ~15 s of evidence in an hour-long video, **what fraction of a 64-frame uniform skim
will land inside the evidence span?** Compute it before running: 64 frames over 5,333 s.

In [4]:
cand = [r for r in rows
        if r["evidence"] and r["evidence"][1] > r["evidence"][0]
        and os.path.exists(make_skim_proxy.proxy_path(PROXY_DIR,  r["videoID"], 64))
        and os.path.exists(make_skim_proxy.proxy_path(PROXY_DIR,  r["videoID"], 256))
        and os.path.exists(make_skim_proxy.proxy_path(PROXY_HIRES, r["videoID"], 32))]
ROW = cand[0]

print(json.dumps({k: ROW[k] for k in ("question_id","videoID","question","options",
                                      "answer","task_type","evidence","time_reference",
                                      "localization_trivial")}, indent=1))

E0, E1 = ROW["evidence"]
print(f"\nevidence span   : {E0:.0f}-{E1:.0f}s  (width {E1-E0:.0f}s)")
print(f"prompt the model sees for the stem:\n  {data.format_question(ROW)!r}")

{
 "question_id": "8",
 "videoID": "qAIRFyR6NyQ",
 "question": "What color hat does the protagonist wear the day she leaves her home?",
 "options": [
  "A. Red",
  "B. Blue",
  "C. Black",
  "D. Yellow"
 ],
 "answer": "B",
 "task_type": "entity recognition",
 "evidence": [
  1170.0,
  1185.0
 ],
 "time_reference": "19:30-19:45",
 "localization_trivial": false
}

evidence span   : 1170-1185s  (width 15s)
prompt the model sees for the stem:
  'What color hat does the protagonist wear the day she leaves her home?\nA. Red\nB. Blue\nC. Black\nD. Yellow'


### Interpreting the output

🟩 **RUNTIME** — our sample is:

| field | value |
|---|---|
| `question_id` | **8** |
| `videoID` | `qAIRFyR6NyQ` |
| question | *"What color hat does the protagonist wear the day she leaves her home?"* |
| gold | **B. Blue** |
| `time_reference` | `19:30-19:45` → `evidence = [1170.0, 1185.0]` |
| task_type | entity recognition |

**The span is 15 seconds inside an 89-minute video — 0.28% of the runtime.** Hold that
number. It is the whole problem statement of this project in one figure: the model has to
find 0.28% of a video, and then read a *colour* off it, which needs spatial detail as well
as temporal aim.

This is a question where **both** axes matter — temporal coverage *and* per-frame
resolution. That is exactly the trade-off compression is supposed to let the agent make.

### 🔧 Exercise 2.2
Compute by hand: a 64-frame uniform skim of a 5,333 s video samples one frame every
_____ seconds. Is 15 s of evidence guaranteed to be sampled? We will check your answer
against the real timestamps in §4.

---
# §3 · Stage 3 · Source MP4 → proxy

```text
 MP4
  v
 Proxy   <-- WE ARE HERE
  v
 Loader
  v
 Processor
  v
 ViT
  v
 Merger
  v
 VidCom2
  v
 mRoPE
  v
 LLM
  v
 Answer
```

## What are we trying to learn from this cell?
The physical properties of the file the server will actually decode — and specifically
whether the proxy preserves the **timeline** of the source.

## Why it matters for the experiment
🟦 **SOURCE** — `make_skim_proxy.py:14-27`. Qwen3-VL writes a literal `<t seconds>` marker
before every frame's vision block, and vLLM derives `t` as `frame_index / fps` read off
the *container*. The model aims its crops from those markers. So if the proxy says a
5,333-second video is 32 seconds long, every crop the model requests is wrong by two
orders of magnitude — and **nothing raises**, because `crop_video` opens the real file and
happily returns frames from 0–32 s.

🟩 **RUNTIME** — this is also where resolution is decided *irrevocably*. Whatever the proxy
throws away, no server-side setting can recover.

## What is entering this step?
The 232 MB source mp4; out comes a ~5 MB proxy written by
`ffmpeg -vf fps=N/duration,scale=…`.

## Before you run this — predict
The 64-frame proxy holds 64 frames. **What should `ffprobe` report as its duration?**
The naive answer (64 frames at a normal frame rate ≈ 2 s) is the failure mode. What must
it be instead, for the timestamps to be right?

In [5]:
def ffprobe(p):
    out = subprocess.run(
        ["ffprobe","-v","error","-select_streams","v:0","-of","json","-show_entries",
         "stream=width,height,r_frame_rate,avg_frame_rate,nb_frames,codec_name:format=duration,size",p],
        capture_output=True, text=True, check=True).stdout
    d = json.loads(out); st = d["streams"][0]
    return {"w":int(st["width"]), "h":int(st["height"]), "codec":st["codec_name"],
            "r_frame_rate":st["r_frame_rate"], "avg_frame_rate":st["avg_frame_rate"],
            "nb_frames":int(st.get("nb_frames") or 0),
            "duration":float(d["format"]["duration"]),
            "size_mb":round(float(d["format"]["size"])/1e6,1)}

PATHS = {
 "source          ": ROW["video_path"],
 "proxy n=64      ": make_skim_proxy.proxy_path(PROXY_DIR,   ROW["videoID"], 64),
 "proxy n=256     ": make_skim_proxy.proxy_path(PROXY_DIR,   ROW["videoID"], 256),
 "proxy n=32 hires": make_skim_proxy.proxy_path(PROXY_HIRES, ROW["videoID"], 32),
}
INFO = {k: ffprobe(v) for k, v in PATHS.items()}
src = INFO["source          "]
print(f"{'file':18} {'WxH':>10} {'frames':>8} {'dur(s)':>9} {'MB':>7} "
      f"{'r_frame_rate':>14} {'avg_frame_rate':>22}  drift")
for k, i in INFO.items():
    drift = abs(i["duration"] - src["duration"]) / src["duration"]
    print(f"{k:18} {i['w']}x{i['h']:<5} {i['nb_frames']:>8} {i['duration']:>9.0f} "
          f"{i['size_mb']:>7.1f} {i['r_frame_rate']:>14} {i['avg_frame_rate']:>22}  "
          f"{drift:6.3%}")

file                      WxH   frames    dur(s)      MB   r_frame_rate         avg_frame_rate  drift
source             1280x720     159829      5333   232.0     30000/1001             30000/1001  0.000%
proxy n=64         448x250         64      5333     1.3         125/12                64/5333  0.000%
proxy n=256        448x250        256      5333     5.0         125/12     23040000/479970001  0.000%
proxy n=32 hires   1280x720         32      5333     2.1           32/1      2880000/479970001  0.001%


### Interpreting the output

🟩 **RUNTIME**:

| file | W×H | frames | duration | size |
|---|---|---|---|---|
| source | 1280×720 | 159,829 | 5,333 s | 232.0 MB |
| proxy n=64 | **448×250** | 64 | **5,333 s** | 1.3 MB |
| proxy n=256 | **448×250** | 256 | **5,333 s** | 5.0 MB |
| proxy n=32 hires | **1280×720** | 32 | **5,333 s** | 2.1 MB |

**The timeline survived — every proxy reports 5,333 s, drift 0.00%.** That is the check
that matters, and it is why `build_one` uses a single `fps=n/duration` filter: 🟦 **SOURCE**
`make_skim_proxy.py:57-64` — that one filter both *picks* n uniform frames and *stamps*
them at that rate, so the output duration comes back out as `n / (n/duration) = duration`.

**Now the part that is easy to miss.** Compare the last two columns. The 64-frame proxy's
`avg_frame_rate` is `64/5333` ≈ 0.012 fps — correct, and exactly what the timeline trick is
supposed to produce. The 256-frame proxy's `r_frame_rate` tag reads `23040000/479970001`,
which is the *nominal* container rate and a different number entirely.

Two rates, one file. **Which one does vLLM read?** If it took the nominal tag, every
`<t seconds>` marker would be wrong — and nothing would raise. §4 answers this by printing
the value vLLM's own loader produced.

**And look at the resolution column.** 🟩 **RUNTIME** — the standard proxies are **448×250**.
🟦 **SOURCE** — `make_skim_proxy.py:133-139` is candid about why: `--max-side 448` is
*inherited* from LongVT, which ships up to 768 frames as images and therefore needs the
per-frame cost tiny. Our experiment has 64–256 frames and a 40,960-token context. The
constraint was copied, not chosen.

> 🟧 **OURS** — **This is a hard ceiling.** 448×250 = 112,000 source pixels per frame. No
> `max_pixels`, no `size` override, no server flag can put back detail the proxy already
> discarded. §5 will show a config that *looks* like it raises resolution and provably
> does not.

### 🔧 Exercise 3
Open `make_skim_proxy.py::verify_one` and read the three conditions it checks. Then write
a one-line answer: *which of those three conditions, if it silently failed, would produce
wrong crop spans rather than a crash?*

---
# §4 · Stage 5–6 · vLLM's video loader and the timestamps

```text
 MP4
  v
 Proxy
  v
 Loader   <-- WE ARE HERE
  v
 Processor
  v
 ViT
  v
 Merger
  v
 VidCom2
  v
 mRoPE
  v
 LLM
  v
 Answer
```

## What are we trying to learn from this cell?
Two things: (a) the frame array vLLM actually builds, and (b) the `<t seconds>` markers it
will write into the prompt — computed with the *same function the server uses*.

## Why it matters for the experiment
🟦 **SOURCE** — `qwen3_vl.py:1069` calls `_get_video_second_idx`, which calls
`_calculate_timestamps` (l. 814). The model reads those markers and copies them into
`crop_video(start_time=…)`. In §11 you will see our sample's model do exactly that,
literally. So these floats are not decoration — they are the *coordinate system* the agent
navigates in.

## Where are we?
Stage 5–6. The mp4 has been decoded to a numpy array; the processor has not run yet.

## What is entering this step?
Proxy bytes + `--media-io-kwargs '{"video":{"num_frames":N}}'`.
🟦 **SOURCE** — `vllm/multimodal/video.py:359-390`, `OpenCVVideoBackend.load_bytes`.

## Before you run this — predict
1. The 64-frame proxy contains exactly 64 frames and we ask for 64. **How many will the
   loader return?**
2. `_calculate_timestamps` averages indices in pairs. **For a 64-frame input, how many
   timestamps come out?**
3. Recall §2.2: is 15 s of evidence at 1,170 s going to be sampled by a 64-frame skim?

In [6]:
from vllm.multimodal.video import OpenCVVideoBackend

def calculate_timestamps(indices, video_fps, merge_size):
    """VERBATIM from vllm/model_executor/models/qwen3_vl.py:814-827."""
    if len(indices) % merge_size != 0:
        indices = indices + [indices[-1]] * (merge_size - len(indices) % merge_size)
    ts = [idx / video_fps for idx in indices]
    return [(ts[i] + ts[i + merge_size - 1]) / 2 for i in range(0, len(ts), merge_size)]

def load(path, num_frames):
    frames, meta = OpenCVVideoBackend.load_bytes(open(path, "rb").read(),
                                                 num_frames=num_frames)
    ts = calculate_timestamps(list(meta["frames_indices"]), meta["fps"], TPS)
    return frames, meta, ts

for label, path, nf in [("64f ", PATHS["proxy n=64      "], 64),
                        ("256f", PATHS["proxy n=256     "], 256),
                        ("32hi", PATHS["proxy n=32 hires"], 32)]:
    frames, meta, ts = load(path, nf)
    step = (ts[-1] - ts[0]) / (len(ts) - 1)
    print(f"[{label}] frames array {str(frames.shape):22} dtype={frames.dtype}")
    print(f"       loader fps={meta['fps']:.6f}  duration={meta['duration']:.0f}s  "
          f"total_num_frames={meta['total_num_frames']}  sampled={len(meta['frames_indices'])}")
    print(f"       timestamps: n={len(ts)}  first={ts[0]:.1f}  step={step:.2f}s  last={ts[-1]:.1f}")
    inside = [(k, round(t,1)) for k, t in enumerate(ts) if E0 <= t <= E1]
    print(f"       markers INSIDE the evidence span [{E0:.0f},{E1:.0f}]: {inside if inside else 'NONE'}")
    print()

[64f ] frames array (64, 250, 448, 3)      dtype=uint8
       loader fps=0.012001  duration=5333s  total_num_frames=64  sampled=64
       timestamps: n=32  first=41.7  step=166.66s  last=5208.0
       markers INSIDE the evidence span [1170,1185]: NONE

[256f] frames array (256, 250, 448, 3)     dtype=uint8
       loader fps=0.048003  duration=5333s  total_num_frames=256  sampled=256
       timestamps: n=128  first=10.4  step=41.66s  last=5301.8
       markers INSIDE the evidence span [1170,1185]: [(28, 1177.0)]

[32hi] frames array (32, 720, 1280, 3)     dtype=uint8
       loader fps=0.006000  duration=5333s  total_num_frames=32  sampled=32
       timestamps: n=16  first=83.3  step=333.31s  last=5083.0
       markers INSIDE the evidence span [1170,1185]: NONE



### Interpreting the output — and a discrepancy worth stopping on

🟩 **RUNTIME**:

| skim | frames array | loader fps | timestamps | step | markers inside evidence |
|---|---|---|---|---|---|
| 64f | `(64, 250, 448, 3)` | 0.012001 | 32 | 166.66 s | **NONE** |
| 256f | `(256, 250, 448, 3)` | 0.048003 | 128 | 41.66 s | **k=28 → 1177.0 s** ✅ |
| 32f hires | `(32, 720, 1280, 3)` | 0.006000 | 16 | 333.4 s | **NONE** |

Three things to take from this.

**(1) 64 raw frames → 32 timestamps.** `expected != observed` only if you forgot
`temporal_patch_size=2`. This is not "half the frames disappeared". 🟦 **SOURCE** —
`_calculate_timestamps` averages indices *in pairs*:

```python
ts = [idx / video_fps for idx in indices]                    # 64 values
return [(ts[i] + ts[i + merge_size - 1]) / 2                 # -> 32 values
        for i in range(0, len(ts), merge_size)]
```

Two raw frames become one temporal grid step, and its timestamp is the **mean** of the
pair's times. Every downstream token count is per *grid* frame, never per raw frame. §8
makes this concrete in tensor shapes.

**(2) The loader read the *real* low fps, not the container's nominal tag.** 🟩 **RUNTIME** —
`fps=0.012001` for the 64-frame proxy, i.e. `64/5333`. 🟦 **SOURCE** —
`video.py:140-144` uses `cap.get(cv2.CAP_PROP_FPS)`, which returns the true average rate,
not the `r_frame_rate` tag we saw in §3. So `duration = 64 / 0.012001 = 5333 s` ✅. The
proxy trick works, and now you have seen the exact line that makes it work.

**(3) 🟩 RUNTIME — This is the finding that justifies the whole research direction.**

> The 64-frame skim **never samples the evidence**. Its markers straddle the span:
> k=6 at **1041.6 s** and k=7 at **1208.3 s**, with the answer sitting at 1170–1185 s in
> between. The model is being asked what colour a hat is, using a set of frames that does
> not contain the hat.
>
> The 256-frame skim **does** sample it: marker k=28 at **1177.0 s** falls inside
> [1170, 1185].

That is *temporal coverage* stated as a measurement rather than an intuition. Remember it
— in §11 we check what each arm actually answered, and the result is not subtle.

### 🔧 Exercise 4
Predict, then verify with one line of arithmetic: **how many frames would a uniform skim
need for its worst-case gap to be under 15 seconds on a 5,333 s video?** Compare that to
the 256 we use. What does your answer say about whether "more frames" alone can solve
localization?

---
# §5 · Stage 7 · The processor, `grid_thw`, and the `max_pixels` trap

```text
 MP4
  v
 Proxy
  v
 Loader
  v
 Processor   <-- WE ARE HERE
  v
 ViT
  v
 Merger
  v
 VidCom2
  v
 mRoPE
  v
 LLM
  v
 Answer
```

This is the most important section in the notebook. It is where "what the config says"
and "what the runtime did" come apart, and it explains two of the mistakes on your list at
once.

## §5.1 · The real `grid_thw`

## What are we trying to learn?
The actual `video_grid_thw` that vLLM's *own* multimodal processor produces — plus the
actual number of `<|video_pad|>` tokens it reserves in the prompt.

## Why it matters
Every token-budget statement in `RESEARCH.md` is downstream of this tensor. And the
`<|video_pad|>` count is the honest measurement: it is what the LLM will actually spend
context on, counted from token ids rather than derived from a formula.

## Where are we?
Stage 7–8. Frames in, patches and placeholders out. The ViT has still not run.

## What is entering this step?
The `(N,H,W,3)` array and metadata from §4, plus `mm_processor_kwargs` and
`video_pruning_rate` exactly as `serve_arm.sh` passes them.

## One implementation note you need in order to trust this cell
🟦 **SOURCE** — `vllm/multimodal/processing/processor.py:1276-1288`,
`_apply_hf_processor_main` branches on `isinstance(prompt, str)`. The **str** branch lets
the HF processor expand the placeholder itself, and vLLM 0.19 then cannot locate the
pruning-shortened run (it raises `Expected there to be 1 prompt placeholders … found 0`).
The OpenAI server **tokenizes first**, so the token-ids branch is the production path.
We pass token ids below for exactly that reason.

## Before you run this — predict
For the 256-frame 448×250 proxy:
- `grid_t` = ? (you know this from §4)
- the frame is 250×448; patches are 16 px and must be a multiple of 32 px after merging.
  **Guess `grid_h`, `grid_w`, and hence tokens per grid-frame.**
- total visual tokens = ?

Write your three numbers down now.

In [7]:
from vllm.config import ModelConfig
from vllm.multimodal import MULTIMODAL_REGISTRY
from vllm.multimodal.processing.inputs import ProcessorInputs
from vllm.multimodal.processing.context import TimingContext

PROMPT = ("<|im_start|>user\n<|vision_start|><|video_pad|><|vision_end|>"
          "What colour hat?<|im_end|>\n<|im_start|>assistant\n")

def process(path, num_frames, video_pruning_rate=None, mm_processor_kwargs=None):
    """Drive vLLM's REAL Qwen3-VL multimodal processor. No GPU, no weights."""
    mc = ModelConfig(model=MODEL, tokenizer=MODEL, dtype="bfloat16", max_model_len=40960,
                     limit_mm_per_prompt={"image": 768, "video": 2},
                     mm_processor_kwargs=mm_processor_kwargs or {},
                     media_io_kwargs={"video": {"num_frames": num_frames}},
                     video_pruning_rate=video_pruning_rate, enforce_eager=True)
    p = MULTIMODAL_REGISTRY.create_processor(mc)
    frames, meta = OpenCVVideoBackend.load_bytes(open(path, "rb").read(),
                                                 num_frames=num_frames)
    items = p.data_parser.parse_mm_data({"video": [(frames, meta)]})
    tok = p.info.get_tokenizer()
    out = p.apply(ProcessorInputs(prompt=tok.encode(PROMPT), mm_data_items=items),
                  TimingContext(enabled=False))          # token ids -- see note above
    ids  = out["prompt_token_ids"]
    item = out["mm_kwargs"]["video"][0]
    g    = item["video_grid_thw"].data
    T, H, W = [int(x) for x in (g[0] if g.ndim == 2 else g)]
    tpf  = (H // MERGE) * (W // MERGE)
    kept = sum(1 for i in ids if i == VIDEO_TOKEN_ID)
    return dict(T=T, H=H, W=W, tpf=tpf, pre=T*tpf, kept=kept, ratio=kept/(T*tpf),
                per_frame=kept/T, prompt_tokens=len(ids),
                px=(H*PATCH, W*PATCH), pv=tuple(item["pixel_values_videos"].data.shape))

m = process(PATHS["proxy n=256     "], 256)
print("256-frame skim, no pruning")
print(f"  video_grid_thw       = ({m['T']}, {m['H']}, {m['W']})")
print(f"  pixel_values_videos  = {m['pv']}")
print(f"  resized frame        = {m['px'][0]} x {m['px'][1]} px")
print(f"  pre-merger patches/grid-frame  = H*W       = {m['H']*m['W']}")
print(f"  post-merger tokens/grid-frame  = (H/2)(W/2) = {m['tpf']}")
print(f"  total visual tokens            = T*tpf      = {m['pre']:,}")
print(f"  <|video_pad|> reserved in ids               = {m['kept']:,}")
print(f"  TOTAL prompt tokens (video + 12 words)      = {m['prompt_tokens']:,}")

256-frame skim, no pruning
  video_grid_thw       = (128, 14, 26)
  pixel_values_videos  = (46592, 1536)
  resized frame        = 224 x 416 px
  pre-merger patches/grid-frame  = H*W       = 364
  post-merger tokens/grid-frame  = (H/2)(W/2) = 91
  total visual tokens            = T*tpf      = 11,648
  <|video_pad|> reserved in ids               = 11,648
  TOTAL prompt tokens (video + 12 words)      = 13,041


### Interpreting the output

🟩 **RUNTIME** — `video_grid_thw = (128, 14, 26)`.

Decompose it, because the numbers are the point:

```text
T = 128   temporal grid steps        = 256 raw frames / temporal_patch_size 2
H = 14    patch rows                 = 224 px / 16
W = 26    patch cols                 = 416 px / 16
```

and therefore

```text
pixel_values_videos = (46592, 1536)
                       ^^^^^  ^^^^
                       |      1536 = 3 channels x 16 x 16 patch x 2 temporal frames
                       |             (a flattened spatio-temporal patch)
                       46592 = T * H * W = 128 * 14 * 26   (pre-merger ViT patches)

tokens/grid-frame  = (14/2) * (26/2) =  7 * 13 =    91
total visual tokens = 128 * 91                 = 11,648
```

**One row of the post-merger tensor = one 32×32-pixel region of one temporal grid step
(= one pair of raw frames).**

Now check your prediction. Two places people usually get it wrong:

- **`T=128`, not 256.** If you predicted 256 you were reasoning in raw frames. Everything
  in this pipeline after the loader is per *grid* frame.
- **The frame was resized from 250×448 to 224×416**, not left alone. That is `smart_resize`
  firing. Which budget made it fire? That is §5.2, and the answer is not the one in
  `serve_arm.sh`.

### 🔧 Exercise 5.1
Without running anything: the 32-frame hi-res proxy is 720×1280. Predict its `grid_thw`
and tokens-per-grid-frame. Then run `process(PATHS["proxy n=32 hires"], 32)` and check.
(Hint: the answer is *not* `720/16 = 45` rows, because `smart_resize` will move it.)

## §5.2 · The `max_pixels` trap

## What are we trying to learn?
Whether `--mm-processor-kwargs '{"max_pixels": …}'` — which appears on **every** line of
`serve_arm.sh` — actually changes anything for video.

## Why it matters for the experiment
Two items on your own list of research-critical mistakes come together here:

> *"assuming `max_pixels` would increase resolution when the proxy had already discarded
> those pixels"*
> *"inheriting LongVT's low-resolution configuration even though our experiment had a very
> different number of frames"*

🟦 **SOURCE** — `serve_arm.sh` sets `MMKW='{"max_pixels":50176,...}'` for arms a–d (LongVT's
224²) and `'{"max_pixels":786432,...}'` for arms e/f, with a comment explaining that e/f
exist to reproduce "VidCom2's own validated setting". If that knob does not do what the
comment says, both the baseline arms *and* the high-res arms are at a different operating
point than recorded.

## The experiment
Sweep the knob across five settings and one alternative spelling (`size`), on identical
input. If the knob works, the grid must change.

## Before you run this — predict
`max_pixels=50176` is 224². `max_pixels=786432` is ~15× larger. Same proxy, same frames.
**How many distinct `grid_thw` values should the sweep produce?** Write down your number.

In [8]:
print("=== 256-frame 448x250 proxy — sweep the max_pixels knob ===")
sweep = [None,
         {},
         {"max_pixels": 50176,  "min_pixels": 3136},     # serve_arm.sh arms a-d
         {"max_pixels": 786432, "min_pixels": 3136},     # serve_arm.sh arms e/f
         {"max_pixels": 200704},
         {"size": {"longest_edge": 786432,   "shortest_edge": 3136}},
         {"size": {"longest_edge": 4_000_000,"shortest_edge": 3136}}]
for k in sweep:
    r = process(PATHS["proxy n=256     "], 256, mm_processor_kwargs=k)
    print(f"  {str(k):<58} -> T={r['T']:>3} H={r['H']:>2} W={r['W']:>2} "
          f"tpf={r['tpf']:>4}  px={r['px'][0]}x{r['px'][1]}")

print("\n=== 32-frame 1280x720 hi-res proxy ===")
for k in [{}, {"max_pixels": 786432, "min_pixels": 3136},
          {"size": {"longest_edge": 786432, "shortest_edge": 3136}}]:
    r = process(PATHS["proxy n=32 hires"], 32, mm_processor_kwargs=k)
    print(f"  {str(k):<58} -> T={r['T']:>3} H={r['H']:>2} W={r['W']:>2} "
          f"tpf={r['tpf']:>4}  px={r['px'][0]}x{r['px'][1]}")

=== 256-frame 448x250 proxy — sweep the max_pixels knob ===


  None                                                       -> T=128 H=14 W=26 tpf=  91  px=224x416


  {}                                                         -> T=128 H=14 W=26 tpf=  91  px=224x416


  {'max_pixels': 50176, 'min_pixels': 3136}                  -> T=128 H=14 W=26 tpf=  91  px=224x416


  {'max_pixels': 786432, 'min_pixels': 3136}                 -> T=128 H=14 W=26 tpf=  91  px=224x416


...

  {'size': {'longest_edge': 786432, 'shortest_edge': 3136}}  -> T=128 H= 2 W= 4 tpf=   2  px=32x64


  {'size': {'longest_edge': 4000000, 'shortest_edge': 3136}} -> T=128 H= 4 W=10 tpf=  10  px=64x160

=== 32-frame 1280x720 hi-res proxy ===


  {}                                                         -> T= 16 H=40 W=72 tpf= 720  px=640x1152


  {'max_pixels': 786432, 'min_pixels': 3136}                 -> T= 16 H=40 W=72 tpf= 720  px=640x1152


  {'size': {'longest_edge': 786432, 'shortest_edge': 3136}}  -> T= 16 H= 6 W=12 tpf=  18  px=96x192


### Interpreting the output — `expected != observed`, and it is not a rounding error

🟩 **RUNTIME**:

| `mm_processor_kwargs` | grid | tokens/frame |
|---|---|---|
| `None` | (128, 14, 26) | 91 |
| `{}` | (128, 14, 26) | 91 |
| `{"max_pixels": 50176}` | (128, 14, 26) | **91** |
| `{"max_pixels": 786432}` | (128, 14, 26) | **91** |
| `{"max_pixels": 200704}` | (128, 14, 26) | **91** |
| `{"size": {"longest_edge": 786432}}` | (128, **2**, **4**) | **2** |
| `{"size": {"longest_edge": 4000000}}` | (128, 4, 10) | 10 |

> 🟩 **RUNTIME — Finding 1. `max_pixels` in `--mm-processor-kwargs` is inert for video in
> vLLM 0.19.** Every value, including no value at all, produces an identical grid. The
> `size` spelling *does* take effect. So every video arm in this project has been running
> at the **checkpoint's default** budget of `size.longest_edge = 25,165,824`, not at
> LongVT's 224² and not at arm e/f's 786432.

Why? 🟦 **SOURCE** — vLLM *does* know how to translate the kwarg: `qwen2_vl.py:849-852`
maps `mm_kwargs["max_pixels"]` onto `size["longest_edge"]`. But that translation lives in
`_get_vision_info`, which vLLM uses to **estimate** token counts for scheduling. The
*actual* processor call passes `mm_kwargs` straight through to
`Qwen3VLVideoProcessor.preprocess`, whose signature takes `size` — and it does not know
the name `max_pixels`. The kwarg is dropped.

Confirm it a third way, from arithmetic rather than from either code path:

In [9]:
from transformers.models.qwen3_vl.video_processing_qwen3_vl import smart_resize

print(f"checkpoint default size = {vp.size}")
print(f"{'config':<22} {'max_pixels':>11} {'->':^4} {'resized':>11} {'tpf':>6}   note")
for nf, h, w, label in [(64, 250, 448, "run1/2  64f"),
                        (256, 250, 448, "run3/4  256f"),
                        (32, 720, 1280, "arm e/f 32f hires")]:
    for mp, note in [(vp.size["longest_edge"], "checkpoint default  <-- what actually fired"),
                     (50176,  "LongVT 224^2 (serve_arm.sh a-d)"),
                     (786432, "arm e/f nominal")]:
        hb, wb = smart_resize(num_frames=nf, height=h, width=w, temporal_factor=TPS,
                              factor=PATCH*MERGE, min_pixels=4096, max_pixels=mp)
        tpf = (hb//PATCH//MERGE) * (wb//PATCH//MERGE)
        print(f"{label:<22} {mp:>11,} {'->':^4} {hb:>4}x{wb:<6} {tpf:>6}   {note}")
    print()

checkpoint default size = {'longest_edge': 25165824, 'shortest_edge': 4096}
config                  max_pixels  ->      resized    tpf   note
run1/2  64f             25,165,824  ->   256x448       112   checkpoint default  <-- what actually fired
run1/2  64f                 50,176  ->    32x32          1   LongVT 224^2 (serve_arm.sh a-d)
run1/2  64f                786,432  ->    64x128         8   arm e/f nominal

run3/4  256f            25,165,824  ->   224x416        91   checkpoint default  <-- what actually fired
run3/4  256f                50,176  ->    32x32          1   LongVT 224^2 (serve_arm.sh a-d)
run3/4  256f               786,432  ->    32x64          2   arm e/f nominal

arm e/f 32f hires       25,165,824  ->   640x1152      720   checkpoint default  <-- what actually fired
arm e/f 32f hires           50,176  ->    32x32          1   LongVT 224^2 (serve_arm.sh a-d)
arm e/f 32f hires          786,432  ->    96x192        18   arm e/f nominal



### Interpreting the output

🟩 **RUNTIME** — hand-recomputing `smart_resize` with the **checkpoint default**
(25,165,824) reproduces the observed grids exactly for all three configs: 256×448 → tpf
112, 224×416 → tpf 91, 640×1152 → tpf 720. The other two budgets do not come close. The
kwarg never fired.

Now read the second column of your own output and notice what it implies.

> 🟩 **RUNTIME — Finding 2. `max_pixels` for video is a WHOLE-VIDEO budget.**
> 🟦 **SOURCE** — `smart_resize` compares `t_bar * h_bar * w_bar` against `max_pixels`,
> where `t_bar` is the frame count. So the budget is shared across frames: **asking for
> more frames automatically lowers per-frame resolution.**

This has three consequences you should be able to state at a whiteboard:

1. **Setting `size.longest_edge = 786432` to "get VidCom2's 720 tokens/frame" would do the
   exact opposite** — 🟩 **RUNTIME** it collapses the hi-res arm to tpf=18. To *get* ~720
   tokens/frame at 32 frames you need `longest_edge ≈ 32 × 640 × 1152 ≈ 23.6 M` — i.e.
   roughly the default that was already there. Arms e/f got their 720 tokens/frame **by
   accident**: hi-res proxy plus the default budget, not because of the flag.

2. **run1 and run3 are not budget-matched.** `RESEARCH.md` records "run1/2/5 = 64 frames →
   2,688 visual tokens · run3/4 = 256 frames × 0.25 → 2,688 (exactly matched, verified)".
   🟩 **RUNTIME** for *this* video: run1 = **3,584** tokens, run3 = **2,912**. Off by 19%,
   and in the direction that disadvantages run3. The mechanism is exactly the one above —
   the 256-frame skim gets tpf 91 while the 64-frame skim gets tpf 112, because the same
   whole-video pixel budget is divided among 4× as many frames.

3. **tokens/frame is video-dependent.** It is a function of aspect ratio and frame count.
   🟧 **OURS** — no single number ("47 tokens/frame", "91 tokens/frame") can be quoted for
   the dataset; `serve_arm.sh`'s comments quote 47 and 49 for arms whose measured values
   here are 112 and 91. Budget claims must be measured per video or reported as a
   distribution.

### 🔧 Exercise 5.2
Two tasks, both by hand.
1. Using `smart_resize`'s formula, work out what `size.longest_edge` you would need to give
   the **256-frame** 448×250 proxy the same tokens-per-frame as the 64-frame one (112).
   Then check whether the 448-wide proxy even *has* that many pixels to give.
2. Explain in two sentences why raising `size.longest_edge` on a 448×250 proxy cannot
   reach 720 tokens/frame no matter how large you set it.

### After this section, I should be able to explain
- why a 256-frame input produces `grid_t = 128`
- exactly where spatial resolution becomes visual-token count
- why proxy resolution caps information *before* any server setting is consulted
- how to compute expected pre-compression tokens by hand
- which runtime value proves the configured frame count reached Qwen (`n_sampled` and `T`)
- why `--mm-processor-kwargs max_pixels` proves nothing about video resolution

---
# §6 · Stage 9–10 · ViT and PatchMerger

```text
 MP4
  v
 Proxy
  v
 Loader
  v
 Processor
  v
 ViT   <-- WE ARE HERE
  v
 Merger
  v
 VidCom2
  v
 mRoPE
  v
 LLM
  v
 Answer
```

## What are we trying to learn from this cell?
The tensor that VidCom2 and EVS actually receive. Everything in §7 and §8 operates on this
one object, so its shape and meaning have to be unambiguous.

## Why it matters for the experiment
🟩 **RUNTIME (below)** — pruning happens **after** the vision encoder. So compression here
saves **LLM** context and **LLM** compute, and saves **zero ViT compute**. If you ever
justify compression on "encoder cost", that is wrong for this pipeline.

## Where are we?
Stage 9–10. Patches went into the ViT; merged tokens are coming out.

## What is entering this step?
`pixel_values_videos` of shape `(T·H·W, 1536)` and `video_grid_thw`.

## What we do to make this cheap
We load **only the visual tower** (~576 M params) rather than the whole 8 B model. That is
enough to reproduce the exact tensor the server's pruning hook sees, and it fits beside
other people's jobs on this shared box.

## Before you run this — predict
For the 256-frame skim we measured `T=128`, `tpf=91`, and `out_hidden_size=4096`.
**What shape comes out of the vision model?** Write down both dimensions and what each
one *means* before running.

In [10]:
from transformers.models.qwen3_vl.modeling_qwen3_vl import Qwen3VLVisionModel
from safetensors import safe_open

DEV = os.environ.get("PROBE_DEV", "cuda:6")     # pick a card with >=8 GB free

idx = json.load(open(os.path.join(MODEL, "model.safetensors.index.json")))["weight_map"]
vis_keys = {k: v for k, v in idx.items() if k.startswith("model.visual.")}
sd = {}
for shard in sorted(set(vis_keys.values())):
    with safe_open(os.path.join(MODEL, shard), framework="pt") as f:
        for k in [k for k, v in vis_keys.items() if v == shard]:
            sd[k[len("model.visual."):]] = f.get_tensor(k)

VIS = Qwen3VLVisionModel(cfg.vision_config)
VIS.load_state_dict(sd, strict=False)
VIS = VIS.to(DEV, dtype=torch.bfloat16).eval()
print(f"visual tower on {DEV}: {sum(p.numel() for p in VIS.parameters())/1e6:.0f}M params")

def embed(path, num_frames):
    """proxy mp4 -> post-PatchMerger embeddings, exactly what the pruning hook receives."""
    frames, meta = OpenCVVideoBackend.load_bytes(open(path, "rb").read(),
                                                 num_frames=num_frames)
    meta2 = {k: v for k, v in meta.items() if k != "do_sample_frames"}
    out = vp(videos=[[frames]], video_metadata=[[meta2]],
             do_sample_frames=False, return_tensors="pt")
    pv, g = out["pixel_values_videos"], out["video_grid_thw"]
    with torch.no_grad():
        e, _deepstack = VIS(pv.to(DEV, torch.bfloat16), grid_thw=g.to(DEV))
    return e, [int(x) for x in g[0]]

E256, G256 = embed(PATHS["proxy n=256     "], 256)
T, H, W = G256
tpf = (H // MERGE) * (W // MERGE)
print(f"\ngrid_thw            = ({T}, {H}, {W})")
print(f"video_embeds        = {tuple(E256.shape)}   dtype={E256.dtype}  device={E256.device}")
print(f"  dim0 {E256.shape[0]:>6} = T * tpf = {T} * {tpf} = {T*tpf}   "
      f"{'MATCH' if E256.shape[0]==T*tpf else 'MISMATCH'}")
print(f"  dim1 {E256.shape[1]:>6} = out_hidden_size (the LLM's hidden width)")
print(f"\nreshaped to (T, tpf, D): {tuple(E256.reshape(T, tpf, -1).shape)}")
print(f"  one row = one {PATCH*MERGE}x{PATCH*MERGE}px region of one temporal grid step")
print(f"          = one 32x32px region of ONE PAIR of raw frames")

visual tower on cuda:3: 576M params



grid_thw            = (128, 14, 26)
video_embeds        = (11648, 4096)   dtype=torch.bfloat16  device=cuda:3
  dim0  11648 = T * tpf = 128 * 91 = 11648   MATCH
  dim1   4096 = out_hidden_size (the LLM's hidden width)

reshaped to (T, tpf, D): (128, 91, 4096)
  one row = one 32x32px region of one temporal grid step
          = one 32x32px region of ONE PAIR of raw frames


### Interpreting the output

🟩 **RUNTIME** — `video_embeds = (11648, 4096)`.

Say it in words, because "11648 × 4096" is not knowledge:

```text
11648 = number of visual tokens the LLM would see if nothing were pruned
      = 128 temporal grid steps x 91 tokens per grid step
 4096 = hidden width of each token (== the LLM's hidden size, out_hidden_size)
```

**One row** = one 32×32-pixel region of the resized frame, at one temporal grid step —
i.e. covering **one pair of raw video frames**. There are 91 such regions per grid step
(7 rows × 13 cols).

Three facts about this tensor that matter for §7:

1. 🟩 **RUNTIME** — it is **post-PatchMerger**: the 46,592 pre-merger ViT patches have
   already been folded 4→1. Both compressors act on the merged tokens, never on raw
   patches.
2. 🟩 **RUNTIME** — the ViT already ran to produce it. **Pruning saves no encoder compute.**
   🟧 **OURS** — for a 256-frame skim resent on every one of up to 5 tool rounds, the ViT
   cost is paid 6 times regardless of the retention rate. If wall-clock is ever the
   argument for compression in this project, it is the wrong argument.
3. 🟦 **SOURCE** — `qwen3_vl.py:1843-1848` splits the encoder output per video item with
   `grid_thw.prod(-1) // merge_size // merge_size`, which is the same `T*tpf` arithmetic.

### 🔧 Exercise 6
Modify the cell yourself: add a line printing `E256.element_size() * E256.nelement() / 1e6`
and state, in one sentence, how many megabytes of activations VidCom2 is asked to score —
and therefore why the compressor's own runtime is negligible next to the ViT's.

---
# §7 · VidCom2, from the implementation

```text
 MP4
  v
 Proxy
  v
 Loader
  v
 Processor
  v
 ViT
  v
 Merger
  v
 VidCom2   <-- WE ARE HERE
  v
 mRoPE
  v
 LLM
  v
 Answer
```

## §7.1 · Where it is called from, and how it gets there at all

## What are we trying to learn?
The exact call site, and the mechanism by which an out-of-tree file in *this repo* replaces
a function inside the installed vLLM wheel.

## Why it matters
🟦 **SOURCE** — vLLM 0.19.0 ships **EVS only**. VidCom2 was merged upstream as PR #47750
but that is `main`-only, in no released wheel. So when a result is labelled "VidCom2", the
only thing making that true is the monkey-patch in
`longvt_compression/vidcom2_vllm/patch.py`. If it silently fails to apply, you get **stock
EVS** and a run labelled VidCom2 — with no error anywhere.

## The call chain
🟦 **SOURCE**:

```text
Qwen3VLForConditionalGeneration.get_multimodal_embeddings   qwen3_vl.py:2406
  -> _postprocess_video_embeds_evs                          qwen3_vl.py:1891
       -> compute_retention_mask(emb, size, spatial_merge_size, q)   l. 1925
              ^^^^^^^^^^^^^^^^^^^^ THIS NAME is what the patch rebinds
       -> emb = emb[retention_mask]                                  l. 1932
       -> _create_final_video_embeddings(...)                        l. 1949
```

## The one subtlety in the patch
🟦 **SOURCE** — `patch.py:8-18`. `qwen3_vl.py:73-76` does
`from vllm.multimodal.evs import compute_retention_mask` **at module scope**, which binds
the *function object* into the `qwen3_vl` namespace at import time. Rebinding
`vllm.multimodal.evs.compute_retention_mask` afterwards would therefore have **no effect
on the model**. The name has to be replaced in every module that imported it.

## Before you run this — predict
The patch rebinds names in three modules. **If you rebound only `vllm.multimodal.evs` and
ran a Qwen3-VL server, what would you observe?** (Careful: the answer is not "an error".)

In [11]:
import importlib, inspect
import vllm.multimodal.evs as evs
import vllm.model_executor.models.qwen3_vl as q3

print("BEFORE patch")
print("  evs.compute_retention_mask   ->", evs.compute_retention_mask.__module__)
print("  qwen3_vl.compute_retention_mask ->", q3.compute_retention_mask.__module__)
print("  same object?", evs.compute_retention_mask is q3.compute_retention_mask,
      " <- this is why rebinding only `evs` would be a no-op")

os.environ["FA_PRUNE_METHOD"] = "vidcom2"
from longvt_compression.vidcom2_vllm import patch
patch.install()

print("\nAFTER patch")
print("  evs.compute_retention_mask      ->", evs.compute_retention_mask.__module__)
print("  qwen3_vl.compute_retention_mask ->", q3.compute_retention_mask.__module__)
print("  original EVS preserved as evs.compute_retention_mask_evs ->",
      evs.compute_retention_mask_evs.__module__)

EVS_MASK = evs.compute_retention_mask_evs           # stock EVS
VC2_MASK = q3.compute_retention_mask                # VidCom2 (what the model now calls)
from vllm.multimodal.evs import compute_retained_tokens_count
print("\nsignature (identical by design, so it drops into the same hook):")
print("  ", inspect.signature(VC2_MASK))

BEFORE patch
  evs.compute_retention_mask   -> vllm.multimodal.evs
  qwen3_vl.compute_retention_mask -> vllm.multimodal.evs
  same object? True  <- this is why rebinding only `evs` would be a no-op
[vidcom2_vllm] VidCom2 retention installed into: qwen3_vl, qwen2_5_vl

AFTER patch
  evs.compute_retention_mask      -> longvt_compression.vidcom2_vllm.retention
  qwen3_vl.compute_retention_mask -> longvt_compression.vidcom2_vllm.retention
  original EVS preserved as evs.compute_retention_mask_evs -> vllm.multimodal.evs

signature (identical by design, so it drops into the same hook):
   (video_embeds: torch.Tensor, video_size_thw: 'torch.LongTensor | tuple[int, int, int]', spatial_merge_size: int, q: float) -> torch.Tensor


### Interpreting the output

🟩 **RUNTIME** — before the patch, `evs.compute_retention_mask` and
`qwen3_vl.compute_retention_mask` are **the same object**. After `patch.install()`,
`qwen3_vl`'s name points at `longvt_compression.vidcom2_vllm.retention` while the original
is preserved as `evs.compute_retention_mask_evs` (which is how we get to A/B them below).

Answer to the prediction: rebinding only `vllm.multimodal.evs` would produce **no error and
no VidCom2** — the model would keep calling the EVS function it captured at import time,
and every result would be labelled VidCom2 while being EVS. 🟦 **SOURCE** — `patch.py:72-77`
raises `RuntimeError` if it patched *no* module, which is the guard against a future vLLM
layout change silently reintroducing exactly this.

**How you verify it in a real run.** 🟦 **SOURCE** — `retention.py:130-139` prints
`[vidcom2_vllm] ACTIVE in pid=…` on the first mask computation, deliberately from inside
whichever process computes it. vLLM runs the model in a **separate EngineCore process**, so
a patch applied in the API process is not evidence the model uses it. That print is.

🟩 **RUNTIME** — from `server_arm_b.log` of the banked run3:

```text
(EngineCore pid=3877676) [vidcom2_vllm] ACTIVE in pid=3877676 (first mask: thw=(128, 12, 28), q=0.75)
```

Note `thw=(128, …)` — the same `T=128` we derived in §5, from the server itself.

### 🔧 Exercise 7.1
Run `grep -c "ACTIVE in pid" /local1/cfyang/hanklin/outputs/lvbench_agent/server_arm_c.log`
and `…server_arm_b.log`. Explain why one is 0 and the other is not, and what that means
about which of the banked runs are genuinely VidCom2.

## §7.2 · The scoring function, line by line

## What are we trying to learn?
What `vidcom2_scores` computes, on what tensor, and what one scalar in its output means.

## Why it matters
🟨 **PAPER** vs 🟦 **SOURCE** disagree here, and the header of `retention.py` says so
explicitly:

> *"Algorithm is lifted from the authors' released reference implementation … NOT from the
> paper — the two disagree, and vLLM upstream also reimplemented from the code because that
> is what produced the published numbers. Specifically the code adds, and the paper never
> mentions: a low-variance channel subset, a sum of 5 Gaussian kernels instead of cosine
> similarity, and centres computed on L2-normalised tokens."*

So: 🟨 the paper describes cosine similarity to a global video representation; 🟦 the code
that ships computes a multi-scale Gaussian distance on half the channels. **Do not quote
the paper's equation when describing what this run did.**

## Reading the code

🟦 **SOURCE** — `longvt_compression/vidcom2_vllm/retention.py:81-107`:

```python
x = video_embeds.reshape(T * tpf, -1).float()          # (11648, 4096)

var  = x.var(dim=0, unbiased=False)                    # (4096,)  variance PER CHANNEL
n_keep = max(1, int(x.shape[-1] * CHANNEL_KEEP_RATIO)) # 2048
chan = torch.topk(var, k=n_keep, largest=False).indices
sel  = x[:, chan].reshape(T, tpf, n_keep)              # (128, 91, 2048)
```

- `x.var(dim=0)` reduces over **all 11,648 tokens**, giving one variance per channel.
- `largest=False` keeps the **lowest**-variance half. 🟨 **PAPER** does not mention this
  step at all. 🟧 **OURS** — the plausible reading is that low-variance channels carry the
  "generic content" signal that a distance-to-centre score wants, while high-variance
  channels are dominated by a few outlier tokens. That is a hypothesis, not a finding.

```python
fr = F.normalize(sel, dim=-1)                          # (T, tpf, C), unit-norm per token
vid_center   = fr.mean(dim=(0, 1), keepdim=True)       # (1, 1, C)  one centre for the video
frame_center = fr.mean(dim=1,      keepdim=True)       # (T, 1, C)  one centre per frame

def _multi_scale_gaussian(centre):
    d2 = ((fr - centre) ** 2).sum(dim=-1)              # (T, tpf) squared L2 distance
    return sum(torch.exp(-d2 / (2 * a)) for a in GAUSSIAN_ALPHAS)   # 5 kernels
```

- `fr - centre` broadcasts `(T,tpf,C) - (1,1,C)` → `(T,tpf,C)`; `.sum(dim=-1)` collapses
  the channel axis, leaving **one scalar per token**.
- `GAUSSIAN_ALPHAS = [2**k for k in range(-3, 2)]` = `[0.125, 0.25, 0.5, 1, 2]` — five
  bandwidths summed, so the score is sensitive at several distance scales at once.
- **High score = close to the centre = typical = redundant.** The mask therefore keeps the
  tokens with the *lowest* score (`largest=False` in the topk at l. 173).

```python
return -v_score.mean(dim=-1), v_score + f_score
#      ^^^^^^^^^^^^^^^^^^^^^  frame_importance (T,)   -- drives the per-frame BUDGET
#                             token_score (T, tpf)    -- drives WHICH tokens survive
```

**One scalar of `token_score[t, i]`** = how typical region *i* of grid-frame *t* is,
relative to (a) the whole video's average token and (b) its own frame's average token.
Note it is **query-agnostic**: the question text never enters this computation. 🟧 **OURS** —
that is a real limitation for a QA task, and it is one of the things a *learned*
compression action could improve on.

## Before you run this — predict
`frame_importance` has one value per grid-frame. It is fed to
`softmax((x - x.max()) / 0.01)`. **With a temperature of 0.01 and 128 frames, how peaked
will that distribution be?** Guess the maximum probability.

In [12]:
import torch.nn.functional as F
from longvt_compression.vidcom2_vllm.retention import (
    vidcom2_scores, _apportion, SOFTMAX_TEMP, GAUSSIAN_ALPHAS, CHANNEL_KEEP_RATIO)

print(f"CHANNEL_KEEP_RATIO={CHANNEL_KEEP_RATIO}  GAUSSIAN_ALPHAS={GAUSSIAN_ALPHAS}  "
      f"SOFTMAX_TEMP={SOFTMAX_TEMP}")

fi, ts = vidcom2_scores(E256, T, tpf)
print(f"\nframe_importance : {tuple(fi.shape)}  min={fi.min():.4f} max={fi.max():.4f} "
      f"std={fi.std():.4f}  SPREAD={float(fi.max()-fi.min()):.4f}")
print(f"token_score      : {tuple(ts.shape)}  min={ts.min():.3f} max={ts.max():.3f}")

probs = F.softmax((fi - fi.max()) / SOFTMAX_TEMP, dim=0)
ent   = -(probs * probs.clamp_min(1e-30).log()).sum()
print(f"\nsoftmax(temp={SOFTMAX_TEMP}): max prob = {probs.max():.6f}, "
      f"2nd = {probs.sort(descending=True).values[1]:.3e}")
print(f"  entropy = {ent:.4f} nats   (a uniform distribution over {T} frames = {np.log(T):.4f})")
print(f"  effective number of frames sharing the bonus = exp(entropy) = "
      f"{float(torch.exp(ent)):.2f}  of {T}")

CHANNEL_KEEP_RATIO=0.5  GAUSSIAN_ALPHAS=[0.125, 0.25, 0.5, 1.0, 2.0]  SOFTMAX_TEMP=0.01

frame_importance : (128,)  min=-2.2275 max=-1.8606 std=0.0538  SPREAD=0.3669
token_score      : (128, 91)  min=3.585 max=5.434

softmax(temp=0.01): max prob = 0.952383, 2nd = 3.345e-02
  entropy = 0.2205 nats   (a uniform distribution over 128 frames = 4.8520)
  effective number of frames sharing the bonus = exp(entropy) = 1.25  of 128


### Interpreting the output

🟩 **RUNTIME** — `frame_importance` spans only **0.367** across all 128 frames
(min −2.2275, max −1.8606, std 0.054). Divide a spread of 0.37 by a temperature of **0.01**
and the softmax sees logits 37 apart. The result:

```text
max prob = 0.952        2nd = 3.3e-02        entropy = 0.22 nats
effective number of frames sharing the budget bonus = 1.25  of 128
```

> 🟩 **RUNTIME — Finding 3. VidCom2's adaptive per-frame budget is effectively degenerate
> at this operating point.** One frame absorbs 95% of the allocation signal; the other 127
> are indistinguishable from each other.

That is a strong claim, so let us watch it turn into actual integer budgets.

## §7.3 · From scores to per-frame budgets

## What are we trying to learn?
The integer number of tokens each grid-frame is allowed to keep, and whether the
distribution is genuinely adaptive.

## Why it matters
"Adaptive per-frame allocation" is 🟨 **PAPER**'s headline distinction from EVS. If it is
flat in practice, then what VidCom2 actually contributes here is its *within-frame* token
scoring, and the story you tell about it must change.

## The hard constraint you must understand first
🟦 **SOURCE** — `retention.py:15-27`. vLLM sizes the `<|video_pad|>` run in the
**processor**, before the model runs, via
`compute_retained_tokens_count(tokens_per_frame, num_frames, q)` (`evs.py:16-35`):

```python
total_tokens   = tokens_per_frame * num_frames
evs_num_tokens = int(total_tokens * (1 - q))
return max(tokens_per_frame, evs_num_tokens)      # never fewer than one frame's worth
```

The mask computed later in the model forward **must produce exactly that many `True`
values**. Off by one and the embeddings no longer line up with the placeholders and
generation is corrupted. 🟦 **SOURCE** — the reference implementation has no such
reconciliation (it rounds each frame's budget independently, so its total drifts), which is
why `retention.py` adds `_apportion` to hit the count exactly while preserving the
allocation *shape*.

## Before you run this — predict
`target = int(128 * 91 * 0.25) = 2912`. With 128 frames that is **22.75 tokens per frame on
average**. Given what §7.2 just showed about the softmax, **predict the histogram of
per-frame budgets**: how many distinct values, and what is the largest?

In [13]:
q = 0.75
target = compute_retained_tokens_count(tokens_per_frame=tpf, num_frames=T, q=q)
print(f"target retained tokens = {target:,} of {T*tpf:,}  ({target/(T*tpf):.4f})")
print(f"                       = {target/T:.2f} tokens per grid-frame on average\n")

base   = 1.0 - q
scales = (base * (1.0 + probs - probs.mean())).clamp(max=1.0)
ideal  = scales * tpf
print(f"scales  : min={scales.min():.4f} median={scales.median():.4f} max={scales.max():.4f}"
      f"   (base = 1-q = {base})")
print(f"ideal/fr: min={float(ideal.min()):.2f} median={float(ideal.median()):.2f} "
      f"max={float(ideal.max()):.2f}")

ks = _apportion(ideal, target=target, lo=1 if target >= T else 0, hi=tpf).cpu()
print(f"\nafter _apportion: sum={int(ks.sum())} (must be exactly {target}) "
      f"min={int(ks.min())} median={int(ks.median())} max={int(ks.max())}")
vals, cnts = np.unique(ks.numpy(), return_counts=True)
print("budget histogram:", "   ".join(f"{v} tok x{c}" for v, c in zip(vals, cnts)))
print(f"\nargmax-importance frame is index {int(fi.argmax())}, which gets "
      f"{int(ks[int(fi.argmax())])} tokens")

target retained tokens = 2,912 of 11,648  (0.2500)
                       = 22.75 tokens per grid-frame on average

scales  : min=0.2480 median=0.2480 max=0.4861   (base = 1-q = 0.25)
ideal/fr: min=22.57 median=22.57 max=44.24

after _apportion: sum=2912 (must be exactly 2912) min=22 median=22 max=91
budget histogram: 22 tok x124   23 tok x1   26 tok x1   44 tok x1   91 tok x1

argmax-importance frame is index 121, which gets 44 tokens


### Interpreting the output

🟩 **RUNTIME**:

```text
scales  : min=0.2480  median=0.2480  max=0.4861     (base = 0.25)
budget histogram: 22 tok x124   23 tok x1   26 tok x1   44 tok x1   91 tok x1
```

**124 of 128 frames receive an identical budget of 22 tokens.** The "adaptive frame budget"
distinguishes four frames out of 128. 🟧 **OURS** — at this operating point VidCom2's
frame-level allocation is doing essentially nothing; what separates it from EVS is the
within-frame token scoring, applied under a near-uniform budget. Any narrative that credits
VidCom2's *frame allocation* for a result on this setup is unsupported.

**A second thing to notice: one frame got 91 — a full frame — though its `ideal` was at most
44.24.** Where did that come from? 🟦 **SOURCE** — `retention.py:58-78`, `_apportion`
water-fills in remainder-priority order and gives each frame its **whole headroom**
(`room = (hi - k)[order]`), not `+1`:

```python
n_full = int((csum <= diff).sum())        # frames that absorb their ENTIRE room
step[order[:n_full]] = room[:n_full]
```

Textbook largest-remainder (Hamilton) apportionment gives each unit either `floor` or
`floor+1`. This gives the top-remainder frame everything it can hold. 🟧 **OURS** — the
result still satisfies the hard constraint (`sum == target`, which is non-negotiable), but
it distorts the shape: one arbitrary frame receives 4× its intended budget, taken from the
rest. Worth flagging as an implementation deviation from *both* the paper and the reference.

### 🔧 Exercise 7.3
Change `SOFTMAX_TEMP` locally (do **not** edit `retention.py` — bind a local variable and
recompute `probs`) to `0.1` and to `1.0`, and re-plot the budget histogram. At which
temperature does frame allocation become genuinely adaptive? Then answer: *is that a bug in
our port, or faithful to the reference implementation?* (Check `retention.py:33-36`.)

## §7.4 · The mask, and the count reconciliation

## What are we trying to learn?
The final boolean mask, its `True` count versus the processor's prediction, and the
distribution of surviving tokens per frame.

## Why it matters
This is the number that answers "did `r=0.25` mean what I thought it meant?" — and, in
§9, the number that shows two experiments both called "25% retention" are at radically
different operating points.

## Before you run this — predict
`mask.sum()` must equal 2,912 **exactly** (see §7.3). What is the mean surviving tokens per
grid-frame? And — the question that matters — **is ~23 tokens enough to describe a frame?**

In [14]:
mask = VC2_MASK(E256, (T, H, W), spatial_merge_size=MERGE, q=q)
per  = mask.reshape(T, tpf).sum(1).float().cpu()

print(f"mask: shape={tuple(mask.shape)} dtype={mask.dtype}  True={int(mask.sum()):,}")
print(f"  processor predicted {target:,}  ->  "
      f"{'EXACT MATCH (generation is safe)' if int(mask.sum())==target else 'MISMATCH -- generation WILL corrupt'}")
print(f"\nactual retention = {int(mask.sum())/(T*tpf):.4f}   (requested {1-q})")
print(f"surviving tokens per grid-frame:")
print(f"  min={per.min():.0f}  p25={per.quantile(.25):.0f}  median={per.median():.0f}  "
      f"p75={per.quantile(.75):.0f}  max={per.max():.0f}  mean={per.mean():.2f}  std={per.std():.2f}")
print(f"  frames left with 0 tokens: {int((per==0).sum())}/{T}")

print(f"\nWhat {per.median():.0f} tokens per frame means spatially:")
print(f"  a full grid-frame is {H//MERGE} x {W//MERGE} = {tpf} regions of "
      f"{PATCH*MERGE}x{PATCH*MERGE}px each")
print(f"  keeping {per.median():.0f} of {tpf} = {per.median()/tpf:.1%} of the frame's area survives")
print(f"  i.e. roughly a {math.sqrt(float(per.median())):.1f} x {math.sqrt(float(per.median())):.1f} "
      f"grid of 32x32px tiles to represent a {H*PATCH}x{W*PATCH}px frame")

[vidcom2_vllm] ACTIVE in pid=265248 (first mask: thw=(128, 14, 26), q=0.75)


mask: shape=(11648,) dtype=torch.bool  True=2,912
  processor predicted 2,912  ->  EXACT MATCH (generation is safe)

actual retention = 0.2500   (requested 0.25)
surviving tokens per grid-frame:
  min=22  p25=22  median=22  p75=22  max=91  mean=22.75  std=6.39
  frames left with 0 tokens: 0/128

What 22 tokens per frame means spatially:
  a full grid-frame is 7 x 13 = 91 regions of 32x32px each
  keeping 22 of 91 = 24.2% of the frame's area survives
  i.e. roughly a 4.7 x 4.7 grid of 32x32px tiles to represent a 224x416px frame


### Interpreting the output

🟩 **RUNTIME**:

```text
True = 2,912   ==  processor's prediction 2,912       EXACT MATCH
actual retention = 0.2500                              (requested 0.25)
surviving tokens/grid-frame: median 22, mean 22.75, min 22, max 91, std 6.39
frames left with 0 tokens: 0/128
```

The mechanical part is healthy: retention is exactly what was asked for, the count
reconciles, no frame is wiped out.

**Now the part that is not healthy.** 🟩 **RUNTIME** — 22 tokens survive to represent a
224×416-pixel frame. That is 24% of the frame's area, or about a **4.7 × 4.7 grid of
32×32-pixel tiles** for the entire frame.

🟧 **OURS** — return to our sample question: *"What colour hat does the protagonist wear?"*
A hat occupies a small part of a frame. At ~23 surviving tokens per frame the odds that the
hat's tile is among the survivors — chosen by a **query-agnostic** typicality score that has
never seen the word "hat" — are poor. This is the mechanism behind your own note that
`r=0.25` at 91 tokens/frame is nothing like `r=0.25` at 720 tokens/frame. §9 puts both on
one table.

### After this section, I should be able to explain
- exactly what tensor VidCom2 receives: `(T·tpf, 4096)` post-PatchMerger, on GPU
- why it saves **no** ViT compute
- how the frame budget is computed — and why it is near-uniform in practice
- how token uniqueness is computed (multi-scale Gaussian distance to two centres, on the
  low-variance half of the channels), and that it is **query-agnostic**
- what `r=0.25` means: `int(T · tpf · 0.25)` tokens survive, reconciled exactly against the
  placeholder run
- why `r=0.25` does **not** imply two configurations carry equivalent information
- how to verify VidCom2 actually ran: the `ACTIVE in pid=` line in the **EngineCore** log

---
# §8 · EVS, and how it differs

```text
 MP4
  v
 Proxy
  v
 Loader
  v
 Processor
  v
 ViT
  v
 Merger
  v
 VidCom2   <-- WE ARE HERE
  v
 mRoPE
  v
 LLM
  v
 Answer
```

## What are we trying to learn?
EVS's scoring, in the same detail — and then a direct A/B of the two masks on identical
embeddings.

## Why it matters
🟦 **SOURCE** — EVS is what runs when the VidCom2 patch does not apply. Knowing its
signature *behaviour* (not just its name) is how you would notice from the outputs alone
that you were running the wrong compressor.

## The code, line by line

🟦 **SOURCE** — `vllm/multimodal/evs.py:59-92`:

```python
video_embeds = video_embeds.reshape(T, H // sms, W // sms, D)      # (128, 7, 13, 4096)

similarity = torch.nn.functional.cosine_similarity(
    video_embeds[1:,  ...],          # frames 1..127   -> (127, 7, 13, 4096)
    video_embeds[:-1, ...],          # frames 0..126   -> (127, 7, 13, 4096)
    dim=-1,                          # reduce the CHANNEL axis
)                                    # -> (127, 7, 13)
dissimilarity = 1 - similarity
```

Work through the slicing, because this is the heart of EVS:

- **Shape before the line:** `(T, h, w, D)` = `(128, 7, 13, 4096)`.
- **`[1:]` and `[:-1]` have compatible shapes** because both drop exactly one frame — one
  from the front, one from the back — leaving `(127, 7, 13, 4096)` each.
- **Which tokens are compared?** Element `[t, i, j]` of the result compares
  `video_embeds[t+1, i, j]` with `video_embeds[t, i, j]` — the **same spatial position**
  `(i, j)` at **adjacent time steps**. It is a *per-location temporal difference*, never a
  comparison between different regions of the same frame.
- **`dim=-1`** reduces over the 4096 channels, so each cosine is between two 4096-vectors.
- **Resulting shape** `(127, 7, 13)`; **one scalar** = "how much did this 32×32 patch of the
  frame change since the previous grid step".
- **Why this is temporal-change-based, not query-aware:** the only input is the video's own
  embeddings. A static background scores ~0 (redundant, dropped); motion scores high (kept).
  The question is never consulted — same blind spot as VidCom2, by a different route.

```python
dissimilarity = torch.cat(
    [255 * torch.ones_like(video_embeds[:1, :, :, 0]), dissimilarity], dim=0)
```

🟦 **SOURCE** — frame 0 has no predecessor, so it is given a sentinel score of **255**,
guaranteeing **all of frame 0 survives**. Watch for this in the output below.

```python
order = torch.argsort(dissimilarity_flat, descending=True, stable=True)
topk_indices = order[:retain_num_tokens]          # ONE GLOBAL top-K over (T,h,w)
```

🟦 **SOURCE** — the selection is a **single global argsort over all T·h·w scores**. There is
no per-frame budget at all: a high-motion stretch of the video can consume the whole budget
and leave later frames with almost nothing.

## The structural contrast

```text
EVS:      same spatial token across neighbouring time steps
          -> cosine difference
          -> ONE GLOBAL top-K            (no per-frame budget; frame 0 forced)

VidCom2:  distance to video centre + distance to frame centre  (Gaussian, low-var channels)
          -> per-frame budget from a softmax over frame importance
          -> per-frame top-K within that budget
```

## Before you run this — predict
Both masks keep exactly 2,912 tokens. **Which will have the higher variance in
tokens-per-frame, and why?** And: **how much do you expect the two keep-sets to overlap?**
If the two methods were selecting the same information you would expect high overlap;
if they measure genuinely different things, low. Guess a percentage.

In [15]:
m_evs = EVS_MASK(E256, (T, H, W), spatial_merge_size=MERGE, q=q)
m_vc2 = VC2_MASK(E256, (T, H, W), spatial_merge_size=MERGE, q=q)

for name, m in (("EVS", m_evs), ("VidCom2", m_vc2)):
    p = m.reshape(T, tpf).sum(1).float().cpu()
    print(f"{name:8} True={int(m.sum()):>6,}  per-frame: min={p.min():>3.0f} "
          f"p25={p.quantile(.25):>3.0f} med={p.median():>3.0f} p75={p.quantile(.75):>3.0f} "
          f"max={p.max():>3.0f}  mean={p.mean():.2f}  std={p.std():>6.2f}")
    print(f"{'':8} frames at 0 tokens: {int((p==0).sum()):>3}/{T}   "
          f"frames at FULL {tpf}: {int((p==tpf).sum())}/{T}")

inter = int((m_evs & m_vc2).sum())
print(f"\nEVS ∩ VidCom2 = {inter:,} of {int(m_evs.sum()):,} "
      f"({inter/int(m_evs.sum()):.1%} of EVS's keep set)")
print(f"random-chance overlap would be ~{int(m_evs.sum())*float(q_:=0.25):.0f} "
      f"({0.25:.0%}) if the two chose independently")

pe = m_evs.reshape(T, tpf).sum(1).float().cpu()
pv2 = m_vc2.reshape(T, tpf).sum(1).float().cpu()
print(f"\nEVS  keeps ALL of frame 0? {int(pe[0])}/{tpf} -> "
      f"{'yes (the 255 sentinel)' if int(pe[0])==tpf else 'no'}")
print(f"VidCom2 frame 0: {int(pv2[0])}/{tpf}  (no such rule)")
print(f"\nEVS per-frame range: {int(pe.min())}..{int(pe.max())}   "
      f"VidCom2 per-frame range: {int(pv2.min())}..{int(pv2.max())}")

EVS      True= 2,912  per-frame: min=  5 p25= 15 med= 20 p75= 26 max= 91  mean=22.75  std= 13.47
         frames at 0 tokens:   0/128   frames at FULL 91: 1/128
VidCom2  True= 2,912  per-frame: min= 22 p25= 22 med= 22 p75= 22 max= 91  mean=22.75  std=  6.39
         frames at 0 tokens:   0/128   frames at FULL 91: 1/128

EVS ∩ VidCom2 = 1,144 of 2,912 (39.3% of EVS's keep set)
random-chance overlap would be ~728 (25%) if the two chose independently

EVS  keeps ALL of frame 0? 91/91 -> yes (the 255 sentinel)
VidCom2 frame 0: 26/91  (no such rule)

EVS per-frame range: 5..91   VidCom2 per-frame range: 22..91


### Interpreting the output

🟩 **RUNTIME**:

| | True | per-frame min / median / max | std | frame 0 |
|---|---|---|---|---|
| EVS | 2,912 | 5 / 20 / **91** | **13.47** | **91 (full)** |
| VidCom2 | 2,912 | 22 / 22 / 91 | **6.39** | 26 |

**EVS ∩ VidCom2 = 1,144 tokens = 39.3% of EVS's keep set** — barely above the ~25% you
would get by chance.

Three readings:

1. **EVS has twice the per-frame variance** (std 13.5 vs 6.4) and its minimum drops to 5
   tokens for a whole frame. That is the global top-K at work: no frame is protected. 🟧
   **OURS** — for a localization task that is a real risk, since a quiet-but-important
   moment (a woman putting on a hat, in a static shot) is exactly what a motion-based score
   discards.
2. **EVS keeps all 91 tokens of frame 0**, VidCom2 keeps 26. 🟦 **SOURCE** — the `255 *
   torch.ones_like(...)` sentinel. If you ever see a run where frame 0 is fully intact and
   frame budgets are wildly uneven, you are running EVS regardless of the label on the run.
3. **The two agree on under 40% of tokens.** These are genuinely different selections, not
   two spellings of the same idea. 🟧 **OURS** — that makes EVS a *useful control*, not a
   substitute: an A/B between them isolates "does *this particular* selection rule matter",
   separately from "does compression at this budget hurt".

### 🔧 Exercise 8
Write ~10 lines yourself: for each grid-frame, compute the *rank* of its EVS budget and its
VidCom2 budget, and print the Spearman correlation. Then answer — do the two methods at
least agree on *which frames* deserve tokens, even when they disagree on which tokens?

---
# §9 · The comparison table

```text
 MP4
  v
 Proxy
  v
 Loader
  v
 Processor
  v
 ViT
  v
 Merger
  v
 VidCom2
  v
 mRoPE
  v
 LLM   <-- WE ARE HERE
  v
 Answer
```

## What are we trying to learn?
All five experiment settings on one table, with **measured** values, so the phrase
"25% retention" stops being a single idea.

## Why it matters
This is the table that answers your question *"why may two experiments both called 25%
retention be radically different operating points?"* — with numbers, on the same video.

## Before you run this — predict
Fill in the two blanks before running:
- 256 frames × 91 tokens/frame × 0.25 = _____ surviving tokens/frame
- 32 frames × 720 tokens/frame × 0.25 = _____ surviving tokens/frame

Both configurations spend ~2,900 visual tokens. Are they the same experiment?

In [16]:
CONFIGS = [
    ("run1/run2   64f video, no pruning",      PATHS["proxy n=64      "],  64, None),
    ("diag        256f video, no pruning",     PATHS["proxy n=256     "], 256, None),
    ("run3/run4   256f video, VidCom2 r=0.25", PATHS["proxy n=256     "], 256, 0.75),
    ("arm e       32f HI-RES, no pruning",     PATHS["proxy n=32 hires"],  32, None),
    ("arm f       32f HI-RES, VidCom2 r=0.25", PATHS["proxy n=32 hires"],  32, 0.75),
    ("TRAP        256f 448-proxy + max_pixels=786432",
                                               PATHS["proxy n=256     "], 256, None),
]
rows_out = []
for label, path, nf, prune in CONFIGS:
    mmk = {"max_pixels": 786432, "min_pixels": 3136} if "TRAP" in label else None
    r = process(path, nf, video_pruning_rate=prune, mm_processor_kwargs=mmk)
    src = ffprobe(path)
    rows_out.append((label, f"{src['w']}x{src['h']}", nf, r))

hdr = (f"{'config':<44} {'proxy':>9} {'raw f':>6} {'gridT':>6} {'px/frame':>10} "
       f"{'tok/f':>6} {'pre':>8} {'post':>8} {'ratio':>7} {'surv/f':>7} {'prompt':>8}")
print(hdr); print("-"*len(hdr))
for label, proxy, nf, r in rows_out:
    print(f"{label:<44} {proxy:>9} {nf:>6} {r['T']:>6} "
          f"{r['px'][0]}x{r['px'][1]:<5} {r['tpf']:>6} {r['pre']:>8,} {r['kept']:>8,} "
          f"{r['ratio']:>7.3f} {r['per_frame']:>7.1f} {r['prompt_tokens']:>8,}")

config                                           proxy  raw f  gridT   px/frame  tok/f      pre     post   ratio  surv/f   prompt
---------------------------------------------------------------------------------------------------------------------------------
run1/run2   64f video, no pruning              448x250     64     32 256x448      112    3,584    3,584   1.000   112.0    3,941
diag        256f video, no pruning             448x250    256    128 224x416       91   11,648   11,648   1.000    91.0   13,041
run3/run4   256f video, VidCom2 r=0.25         448x250    256    128 224x416       91   11,648    2,912   0.250    22.8    4,305
arm e       32f HI-RES, no pruning            1280x720     32     16 640x1152     720   11,520   11,520   1.000   720.0   11,704
arm f       32f HI-RES, VidCom2 r=0.25        1280x720     32     16 640x1152     720   11,520    2,880   0.250   180.0    3,064
TRAP        256f 448-proxy + max_pixels=786432   448x250    256    128 224x416       91   11,64

### Interpreting the output

🟩 **RUNTIME** — every number below is measured on video `qAIRFyR6NyQ`:

| config | proxy | raw f | grid T | px/frame | tok/f | pre | post | ratio | **surv/frame** | prompt |
|---|---|---|---|---|---|---|---|---|---|---|
| run1/run2 64f, no pruning | 448×250 | 64 | 32 | 256×448 | 112 | 3,584 | 3,584 | 1.000 | **112.0** | 3,941 |
| diag 256f, no pruning | 448×250 | 256 | 128 | 224×416 | 91 | 11,648 | 11,648 | 1.000 | **91.0** | 13,041 |
| **run3/run4 256f, VidCom2 r=0.25** | 448×250 | 256 | 128 | 224×416 | 91 | 11,648 | **2,912** | 0.250 | **22.8** | 4,305 |
| arm e 32f hi-res, no pruning | 1280×720 | 32 | 16 | 640×1152 | 720 | 11,520 | 11,520 | 1.000 | **720.0** | 11,704 |
| **arm f 32f hi-res, VidCom2 r=0.25** | 1280×720 | 32 | 16 | 640×1152 | 720 | 11,520 | **2,880** | 0.250 | **180.0** | 3,064 |
| TRAP 256f + `max_pixels=786432` | 448×250 | 256 | 128 | 224×416 | 91 | 11,648 | 11,648 | 1.000 | 91.0 | 13,041 |

**Read the last two lines against the third.**

> 🟩 **RUNTIME** — run3 and arm f both retain **exactly 0.250** and both spend ~2,900 visual
> tokens. They differ by **7.9×** in surviving tokens per frame: **22.8 vs 180.0**.
>
> Same `r`. Same token budget. Not the same experiment.

`r=0.25` is a statement about a *ratio*. What a frame can still describe depends on the
**absolute** number of tokens it has left. 🟨 **PAPER** — VidCom2's near-lossless claim was
measured around the arm-f point (~720 tokens/frame before, ~180 after). 🟧 **OURS** — run3
sits at 22.8, roughly 8× below that. Reporting "VidCom2 at r=0.25 hurt accuracy" without
that qualifier attributes to the compressor what may be an artefact of running it far
outside its validated regime.

**And the TRAP row is your own bullet, verified.** Feeding a 448-wide proxy while asking for
`max_pixels=786432` produces a byte-identical result to asking for nothing: same grid, same
91 tokens/frame, same 13,041 prompt tokens. Two independent reasons, either of which alone
is fatal:
1. §5.2 — `max_pixels` is inert for video in vLLM 0.19.
2. §3 — the proxy holds only 448×250 = 112,000 pixels/frame. **You cannot request pixels
   that were discarded at proxy-build time.**

### 🔧 Exercise 9
Answer in writing, then check against §11's accuracy table: *if run3 loses to run1, name
the three distinct explanations this table admits, and say which single additional run
would separate them.* (Hint: one of the three is already banked.)

---
# §10 · Stage 8, 12–13 · Placeholders, mRoPE, and the sequence the LLM sees

```text
 MP4
  v
 Proxy
  v
 Loader
  v
 Processor
  v
 ViT
  v
 Merger
  v
 VidCom2
  v
 mRoPE   <-- WE ARE HERE
  v
 LLM
  v
 Answer
```

## What are we trying to learn?
How the pruned embeddings get re-attached to the prompt, and what happens to positional
encodings when tokens are deleted from the middle of a sequence.

## Why it matters
🟦 **SOURCE** — `evs.py:154-166`, `recompute_mrope_positions`: *"Original mrope_positions
are computed incorrectly, so once we prune media tokens we should reflect this in the mrope
positions for the LLM."* If this were wrong, the model would still generate — it would just
have an incoherent sense of where in space and time each surviving token came from. That is
a silent-failure mode par excellence.

## The two-stage placeholder mechanism
🟦 **SOURCE** — `qwen3_vl.py:1104-1131` (processor) and `1961-2047` (model forward). The
processor cannot know the mask; the model cannot resize the prompt. So they meet in the
middle:

```text
PROCESSOR (before the model runs, CPU)
  num_tokens = compute_retained_tokens_count(tpf, T, q)        # 2912
  tokens_per_frame = [2912] + [0] * 127        <-- ALL tokens assigned to frame 0
  select_token_id  = False                     <-- placeholders are not "selected"
  -> get_video_repl(...) writes the <|video_pad|> run into the prompt

MODEL FORWARD (GPU, later)
  retention_mask   = compute_retention_mask(emb, ...)          # 2912 True
  emb              = emb[retention_mask]                       # (2912, 4096)
  num_tokens_per_frame = retention_mask.reshape(T,h,w).sum((1,2))   # the REAL per-frame counts
  -> _create_final_video_embeddings rebuilds the repl with the REAL counts
```

🟦 **SOURCE** — `qwen3_vl.py:1111-1114` is explicit about the fiction: *"Here we just need
placeholders that won't actually be replaced — we just need to make sure the total number
of tokens is correct, assign all tokens to the first frame."* The **total** is the contract;
the per-frame distribution is fixed up later.

**This is why `retention.py` must not drift.** 🟦 **SOURCE** — `retention.py:15-27`. Both
sides call the *same* `compute_retained_tokens_count`, which is why `patch.py` deliberately
leaves that function alone and only replaces the mask.

## mRoPE: recomputed from the unpruned grid, then indexed
🟦 **SOURCE** — `qwen3_vl.py:2079-2113`:

```python
unpruned_token_ids = get_video_repl(tokens_per_frame=[tpf]*T, ...)   # FULL grid
original_mrope     = self.get_mrope_input_positions(unpruned_token_ids, ...)
expanded_positions[is_video_embed, :3] = original_mrope[full_is_video_embed][retention_mask]
                                                                            ^^^^^^^^^^^^^^
```

Answering your question directly: positions are **neither** recomputed from the pruned
sequence **nor** naively sliced. They are computed on the **full, unpruned** `(t,h,w)` grid
— so each surviving token keeps its *original* spatial and temporal coordinates — and then
**indexed by the retention mask**. A token that was at grid position (t=93, h=4, w=11) still
reports (93, 4, 11) after 75% of its neighbours are gone.

🟧 **OURS** — this is the right design, and it is worth understanding why: if positions were
renumbered densely after pruning, the model would read a heavily-pruned stretch of video as
temporally compressed, and the `<t seconds>` markers in the text would contradict the
positional encoding.

## Before you run this — predict
The prompt contains a `<|video_pad|>` run plus one `<t seconds>` marker and a
vision_start/vision_end pair **per grid-frame**. For the 256-frame pruned skim:
**how many non-video tokens does the video block cost on top of the 2,912 pads?**

In [17]:
from transformers import AutoTokenizer
tokz = AutoTokenizer.from_pretrained(MODEL)

r_np = process(PATHS["proxy n=256     "], 256, video_pruning_rate=None)
r_pr = process(PATHS["proxy n=256     "], 256, video_pruning_rate=0.75)

print(f"{'':22} {'video pads':>11} {'total prompt':>13} {'overhead':>10}")
for nm, r in (("256f unpruned", r_np), ("256f VidCom2 .25", r_pr)):
    print(f"{nm:22} {r['kept']:>11,} {r['prompt_tokens']:>13,} "
          f"{r['prompt_tokens']-r['kept']:>10,}")

print(f"\nper-grid-frame structural cost = "
      f"{(r_pr['prompt_tokens']-r_pr['kept'])/T:.1f} tokens x {T} grid-frames")
print("  = <t seconds> marker + <|vision_start|> + <|vision_end|>, per frame")
print(f"\ntimestamp marker tokenises as: "
      f"{tokz.encode('<1177.0 seconds>', add_special_tokens=False)} "
      f"-> {len(tokz.encode('<1177.0 seconds>', add_special_tokens=False))} tokens")

# mRoPE geometry on the unpruned grid
from vllm.multimodal.evs import compute_mrope_for_media
pos = compute_mrope_for_media(torch.tensor([T, H, W]), spatial_merge_size=MERGE)
print(f"\ncompute_mrope_for_media -> {tuple(pos.shape)}  = (T*h*w, 4) = "
      f"({T}*{H//MERGE}*{W//MERGE}, 4)")
print("  channels = [t_index, h_index, w_index, llm_grid_w]")
print(f"  first rows:\n{pos[:3]}")
print(f"  t range {int(pos[:,0].min())}..{int(pos[:,0].max())}   "
      f"h range {int(pos[:,1].min())}..{int(pos[:,1].max())}   "
      f"w range {int(pos[:,2].min())}..{int(pos[:,2].max())}")

kept_pos = pos[m_vc2.cpu()]
print(f"\nafter indexing by the VidCom2 mask: {tuple(kept_pos.shape)}")
print(f"  t range still {int(kept_pos[:,0].min())}..{int(kept_pos[:,0].max())} "
      f"-- ORIGINAL coordinates preserved, not renumbered")

                        video pads  total prompt   overhead
256f unpruned               11,648        13,041      1,393
256f VidCom2 .25             2,912         4,305      1,393

per-grid-frame structural cost = 10.9 tokens x 128 grid-frames
  = <t seconds> marker + <|vision_start|> + <|vision_end|>, per frame

timestamp marker tokenises as: [27, 16, 16, 22, 22, 13, 15, 6486, 29] -> 9 tokens

compute_mrope_for_media -> (11648, 4)  = (T*h*w, 4) = (128*7*13, 4)
  channels = [t_index, h_index, w_index, llm_grid_w]
  first rows:
tensor([[ 0,  0,  0, 13],
        [ 0,  0,  1, 13],
        [ 0,  0,  2, 13]])
  t range 0..127   h range 0..6   w range 0..12

after indexing by the VidCom2 mask: (2912, 4)
  t range still 0..127 -- ORIGINAL coordinates preserved, not renumbered


### Interpreting the output

🟩 **RUNTIME**:

```text
256f unpruned      11,648 pads   13,041 prompt   1,393 overhead
256f VidCom2 .25    2,912 pads    4,305 prompt   1,393 overhead
```

**The structural overhead is identical — 1,393 tokens — in both cases.** That is
`1393 / 128 ≈ 10.9` tokens per grid-frame: the `<t seconds>` marker (≈9 tokens) plus
`<|vision_start|>` and `<|vision_end|>`.

🟧 **OURS** — worth noticing: at r=0.25 the *structural* overhead (1,393) is nearly half the
size of the *visual* payload (2,912). Push retention lower and you approach a regime where
most of the video block is frame scaffolding rather than pixels. A useful sanity bound on
how far this direction can be pushed with 128 grid-frames.

🟩 **RUNTIME** — `compute_mrope_for_media` returns `(11648, 4)` with `t` running 0..127,
`h` 0..6, `w` 0..12 — the full unpruned grid. After indexing by the mask the **t range is
still 0..127**: surviving tokens keep their original coordinates. Exactly as the source
said.

### 🔧 Exercise 10
Open `qwen3_vl.py` at line 2110 and explain, in one sentence each, what
`original_mrope[full_is_video_embed][retention_mask]` does in each of its three steps.
Then say what would break if the two boolean masks were applied in the opposite order.

---
# §11 · Stage 14–17 · The agent loop, and what our sample actually did

```text
 MP4
  v
 Proxy
  v
 Loader
  v
 Processor
  v
 ViT
  v
 Merger
  v
 VidCom2
  v
 mRoPE
  v
 LLM
  v
 Answer   <-- WE ARE HERE
```

## What are we trying to learn?
What the model *did* with question 8 in each banked arm — the crop it asked for, the answer
it gave, and why.

## Why it matters
This is where the pipeline stops being tensors and becomes a research result. It is also
the section that makes §4's timestamp finding pay off.

## What is entering this step?
The banked `results.jsonl` and `traj/8.json` from the five runs. 🟦 **SOURCE** —
`run_agent.py:686-699` writes one trajectory per question; `RESEARCH.md` names them the
canonical five.

## Before you run this — predict
From §4 you know the 64-frame skim never samples the evidence and the 256-frame skim does.
**Predict which arms answer question 8 correctly.** Write down five letters.

In [18]:
RUNS = [("run1  64f, no tool",              "lvbench_r1_seed0"),
        ("run2  64f + crop_video",          "lvbench_r2_seed0"),
        ("run3  256f VidCom2 r=.25, no tool","lvbench_c3_notool_seed0"),
        ("run4  256f VidCom2 + crop",       "lvbench_r4_seed0"),
        ("run5  64f + ORACLE crop",         "lvbench_r5_seed0")]

print(f"{'run':36} {'pred':>5} {'gold':>5} {'ok':>4} {'promptTok':>10} {'coverage':>9} {'landed':>7}")
for label, d in RUNS:
    p = f"{OUT}/{d}/results.jsonl"
    row = next((json.loads(l) for l in open(p)
                if l.strip() and json.loads(l)["question_id"] == ROW["question_id"]), None)
    if row is None:
        print(f"{label:36} (absent)"); continue
    print(f"{label:36} {str(row['pred']):>5} {row['gold']:>5} "
          f"{'OK' if row['correct'] else 'X':>4} {row.get('prompt_tokens',0):>10,} "
          f"{str(row.get('covered_frac')):>9} {str(row.get('landed')):>7}")

print("\n--- run2's actual trajectory (the model aiming its own crop) ---")
t = json.load(open(f"{OUT}/lvbench_r2_seed0/traj/{ROW['question_id']}.json"))
for c in t["tool_calls"]:
    print(f"  crop_video({c['start']:.1f}, {c['end']:.1f})  -> {c.get('n_frames')} frames")
print(f"  GT evidence span: [{E0:.0f}, {E1:.0f}]   covered_frac={t['covered_frac']}")

_, _, ts64 = load(PATHS["proxy n=64      "], 64)
hit = [(k, round(x,1)) for k, x in enumerate(ts64) if abs(x - t["tool_calls"][0]["start"]) < 0.5]
print(f"\n  is the model's crop start {t['tool_calls'][0]['start']} a skim TIMESTAMP MARKER? {hit}")

run                                   pred  gold   ok  promptTok  coverage  landed
run1  64f, no tool                       C     B    X      4,038       0.0   False
run2  64f + crop_video                   C     B    X      5,758       0.0   False
run3  256f VidCom2 r=.25, no tool        B     B   OK      4,446       0.0   False
run4  256f VidCom2 + crop                B     B   OK      8,887       0.0   False
run5  64f + ORACLE crop                  C     B    X      5,968       1.0    True

--- run2's actual trajectory (the model aiming its own crop) ---
  crop_video(1041.6, 1069.3)  -> 28 frames
  GT evidence span: [1170, 1185]   covered_frac=0.0

  is the model's crop start 1041.6 a skim TIMESTAMP MARKER? [(6, 1041.6)]


### Interpreting the output — this is the payoff

🟩 **RUNTIME**:

| run | pred | gold | ok | prompt tok | coverage | landed |
|---|---|---|---|---|---|---|
| run1 64f, no tool | C | B | ✗ | 4,038 | 0.0 | False |
| run2 64f + crop | C | B | ✗ | 5,758 | 0.0 | False |
| **run3 256f VidCom2 r=.25, no tool** | **B** | B | **✓** | 4,446 | 0.0 | False |
| **run4 256f VidCom2 + crop** | **B** | B | **✓** | 8,887 | 0.0 | False |
| run5 64f + **oracle** crop | C | B | ✗ | 5,968 | **1.0** | **True** |

And the trajectory:

```text
run2: crop_video(1041.6, 1069.3) -> 28 frames
      GT evidence [1170, 1185]   covered_frac = 0.0
      is 1041.6 a skim timestamp marker?  [(6, 1041.6)]   <-- YES, marker k=6, exactly
```

**Three things, each of which you can now derive rather than believe.**

**(1) The model copies its crop target verbatim from a skim timestamp marker.** 🟩 **RUNTIME**
— `1041.6` is *exactly* marker k=6 of the 64-frame skim. It is not approximately a marker;
it is the marker, to one decimal. So the agent's temporal resolution is **bounded by the
skim's frame spacing** — 166.7 s for a 64-frame skim of this video. Asking it to hit a
15-second window with a 166-second ruler is asking for a 9% chance.

**(2) Both 256-frame arms got it right — run3 with no tool at all.** 🟩 **RUNTIME** — run3
answered **B** correctly at 4,446 prompt tokens, *fewer* than run2's 5,758. run4 (same skim
plus the crop tool) also answered **B**, at 8,887 tokens and with `covered_frac = 0.0` —
i.e. **its crop still missed, and it was right anyway**. The signal came from the skim, not
from the tool.

Why? §4: the 256-frame skim's marker k=28 at **1177.0 s** lands inside the evidence span.
Even at 22.8 surviving tokens per frame, *having sampled the right moment at all* beat
having 112 tokens per frame of the wrong moments.

🟧 **OURS** — that is one question, not evidence. But it is a clean illustration of the
hypothesis the whole chapter is testing: **temporal coverage and per-frame fidelity are
substitutes, and which one binds depends on the question.**

**(3) The oracle failed — and that is informative.** 🟩 **RUNTIME** — run5 forced the crop
onto the evidence span (`covered_frac = 1.0`, `landed = True`), handed the model 128
full-detail frames of exactly the right moment, and it still answered **C**. So for this
question, perfect localization was **not** sufficient. 🟧 **OURS** — either the hat's colour
is not legible at `crop_video`'s 224² per-frame budget, or the model simply misreads it.
That distinction is testable — re-run this one crop at a higher `max_pixels` — and it is
exactly the kind of question you can now ask yourself.

### 🔧 Exercise 11
Open `traj/8.json` for run5 and read the model's `<think>` text. Decide for yourself
whether it *saw* a hat and misnamed the colour, or never located the hat. Then state which
of the two explanations above your reading supports — and what one-line experiment would
settle it.

## §11.2 · Aggregate results, on matched subsets

## What are we trying to learn?
Whether the single-sample story survives at the dataset level — and how easily an
unmatched comparison lies.

## Why it matters
🟦 **SOURCE** — `RESEARCH.md` records that a previous run3 was quoted at 17.8% accuracy
because 223 of its 410 rows were `APIConnectionError` written as `pred=None,
correct=False`. **Always check `sum('error' in r)` before quoting an accuracy.** The cell
below does that first, every time.

The second trap: `diag256` only covers 253 questions. Comparing its 44.66% against run1's
1,549-question 39.77% compares different question sets.

## Before you run this — predict
On the *same* questions: 64 frames uncompressed vs 256 frames uncompressed vs 256 frames
at r=0.25. **Rank them.** Then predict whether the gaps are larger than the ~1 pp you would
expect from T=0.7 sampling noise.

In [19]:
def load_run(d):
    p = f"{OUT}/{d}/results.jsonl"
    if not os.path.exists(p): return None
    rs = [json.loads(l) for l in open(p) if l.strip()]
    return {r["question_id"]: r for r in rs}

ALL = {n: load_run(d) for n, d in [
    ("run1 64f uncompressed",        "lvbench_r1_seed0"),
    ("run2 64f + crop",              "lvbench_r2_seed0"),
    ("run3 256f VidCom2 r=.25",      "lvbench_c3_notool_seed0"),
    ("run4 256f VidCom2 + crop",     "lvbench_r4_seed0"),
    ("run5 64f + oracle crop",       "lvbench_r5_seed0"),
    ("diag 256f UNcompressed",       "lvbench_diag256_seed0"),
    ("arm e 32f HIRES uncompressed", "lvbench_d3_e_nopruning_seed0")]}

print("=== integrity first: never quote an accuracy before this passes ===")
for n, d in ALL.items():
    if d is None: print(f"  {n:32} MISSING"); continue
    err = sum("error" in r for r in d.values())
    none = sum(r.get("pred") is None for r in d.values())
    flag = "  <-- DO NOT QUOTE" if err else ""
    print(f"  {n:32} n={len(d):>5}  error rows={err:>4}  pred=None={none:>3}{flag}")

def acc(d, keys): return 100*sum(bool(d[k]["correct"]) for k in keys)/max(len(keys),1)

print("\n=== A · does temporal coverage help?  (matched on diag256's questions) ===")
sub = set(ALL["run1 64f uncompressed"]) & set(ALL["run3 256f VidCom2 r=.25"]) \
      & set(ALL["diag 256f UNcompressed"])
print(f"  matched subset n = {len(sub)}")
for n in ("run1 64f uncompressed", "diag 256f UNcompressed", "run3 256f VidCom2 r=.25"):
    print(f"    {n:32} {acc(ALL[n], sub):6.2f}%")

print("\n=== B · does per-frame resolution help?  (matched on arm e's questions) ===")
sub2 = set(ALL["run1 64f uncompressed"]) & set(ALL["run3 256f VidCom2 r=.25"]) \
       & set(ALL["arm e 32f HIRES uncompressed"])
print(f"  matched subset n = {len(sub2)}")
for n in ("run1 64f uncompressed", "arm e 32f HIRES uncompressed", "run3 256f VidCom2 r=.25"):
    print(f"    {n:32} {acc(ALL[n], sub2):6.2f}%")

print("\n=== C · the localization gap (full 1,549) ===")
for n in ("run1 64f uncompressed", "run2 64f + crop", "run5 64f + oracle crop"):
    d = ALL[n]
    print(f"    {n:32} {acc(d, set(d)):6.2f}%   n={len(d)}")

=== integrity first: never quote an accuracy before this passes ===
  run1 64f uncompressed            n= 1549  error rows=   0  pred=None=  0
  run2 64f + crop                  n= 1549  error rows=   0  pred=None= 16
  run3 256f VidCom2 r=.25          n= 1549  error rows=   0  pred=None= 18
  run4 256f VidCom2 + crop         n= 1548  error rows=   0  pred=None=  4
  run5 64f + oracle crop           n= 1549  error rows=   0  pred=None=  0
  diag 256f UNcompressed           n=  253  error rows=   0  pred=None=  2
  arm e 32f HIRES uncompressed     n= 1549  error rows=   0  pred=None=  0

=== A · does temporal coverage help?  (matched on diag256's questions) ===
  matched subset n = 253
    run1 64f uncompressed             39.92%
    diag 256f UNcompressed            44.66%
    run3 256f VidCom2 r=.25           36.76%

=== B · does per-frame resolution help?  (matched on arm e's questions) ===
  matched subset n = 1549
    run1 64f uncompressed             39.77%
    arm e 32f HIRES unc

### Interpreting the output

🟩 **RUNTIME** — integrity check: all seven runs report **0 error rows**. The accuracies are
quotable.

**A · Does temporal coverage help? (same 253 questions)**

| | accuracy |
|---|---|
| run1 — 64 frames, uncompressed, 112 tok/frame | 39.92% |
| diag — **256 frames, uncompressed**, 91 tok/frame | **44.66%** |
| run3 — 256 frames, **VidCom2 r=0.25**, 22.8 tok/frame | 36.76% |

> 🟩 **RUNTIME** — 4× the temporal coverage is worth **+4.74 pp**. Compressing that same
> coverage to 25% gives back **−7.90 pp**, landing *below* the 64-frame baseline.

🟧 **OURS** — this is the cleanest statement of the chapter's current result. More frames
help; compression at this operating point costs more than the frames gained. And §9 tells
you the most likely reason: 22.8 surviving tokens per frame is ~8× below where VidCom2 was
validated. **This is a claim about our operating point, not about VidCom2.**

**B · Does per-frame resolution help? (all 1,549 questions)**

| | accuracy |
|---|---|
| run1 — 64 frames @ 112 tok/frame (3,584 tokens) | 39.77% |
| arm e — 32 frames @ **720** tok/frame (11,520 tokens) | 39.70% |
| run3 — 256 frames @ 22.8 tok/frame | 37.96% |

🟩 **RUNTIME** — 6.4× the per-frame resolution and 3.2× the token spend buys **−0.07 pp**,
i.e. nothing. 🟧 **OURS** — combined with A, the emerging picture is that on LVBench
**temporal coverage is the binding constraint and spatial fidelity is not** — at least
until the model has located the right moment. Which is precisely the argument for making
compression an *action*: be coarse and wide while searching, detailed and narrow once
aimed.

**C · The localization gap (full 1,549)**

| | accuracy |
|---|---|
| run1 no tool | 39.77% |
| run2 model aims its own crop | 39.57% |
| **run5 oracle aims the crop** | **55.39%** |

> 🟩 **RUNTIME** — the model's own cropping is worth **−0.20 pp**. Perfect cropping is worth
> **+15.62 pp**. Essentially **all** of the tool's value is locked behind localization the
> model cannot currently do.

🟧 **OURS** — that 15.6 pp is the size of the prize, and §11.1 showed the mechanism: the
agent aims by copying skim timestamp markers, and a 64-frame skim's markers are 166 s
apart. **This is the strongest quantitative argument in the project for denser temporal
skims** — and therefore for making them affordable, which is what compression was supposed
to buy.

### 🔧 Exercise 11.2
`run4` (256f VidCom2 + crop) is in the table above with 1,548 rows. Compute its
matched-subset accuracy against run2 yourself, and then explain why run4 scoring *below*
run1 is consistent with everything in A and C — rather than being a separate mystery.

---
# §12 · Configuration intent vs runtime reality

The single lesson of this notebook, as a table. Left column: what a setting *looks like*
it promises. Right column: what actually determines the outcome, and the one runtime value
that proves it.

| the config says | what it does **not** prove | the runtime value that settles it |
|---|---|---|
| `--frames 256` | that 256 frames reached the model | `len(meta["frames_indices"])` **and** `grid_thw[0]` — which is **128**, because `temporal_patch_size=2` |
| `--media-io-kwargs num_frames` absent | nothing — it fails **open** | vLLM resamples to its own default; a 256-frame request silently costs the same as 64. Check the `non-default args` line in the server log |
| `--mm-processor-kwargs max_pixels=50176` | that frames are 224² | 🟩 **inert for video in vLLM 0.19.** Only `size.longest_edge` moves the grid — and it is a **whole-video** budget. Check `video_grid_thw` |
| `--max-side 448` at proxy-build time | — | a **hard ceiling**. 448×250 = 112,000 px/frame; nothing downstream can exceed it. Check `ffprobe` on the proxy |
| `--video-pruning-rate 0.75` | that 25% survived | count `<\|video_pad\|>` in the prompt ids, or read `prompt_tokens` off the API response |
| `FA_PRUNE_METHOD=vidcom2` | that VidCom2 ran | the `[vidcom2_vllm] ACTIVE in pid=` line, printed from **EngineCore**, not the API process |
| `r = 0.25` | that two configs carry equal information | **surviving tokens per frame**: 22.8 (run3) vs 180.0 (arm f) |
| `--tools crop_video` | that the prompt matches the tools | `--print-prompt`. A no-tool arm handed the tool prompt is not a vanilla baseline |
| `results.jsonl` exists | that its accuracy is real | `sum("error" in r)`. 223 error rows once scored a run at 17.8%, below chance |
| a run directory | that it is clean | `run_agent` **skips** rows already present. A nonzero skip on a fresh run means contamination |
| `curl /v1/models` → 200 | that the server works | the API process can be up while EngineCore is dead. Send a real generation |

### The generalisation worth keeping

Every row has the same shape: **a setting that fails open**. Nothing raises; you get a
number; the number is wrong. The defence is not care — it is measuring the *effect* rather
than asserting the *cause*, and doing it before the run rather than after. That is §13.

---
# §13 · The runtime invariant checklist

A module has been added for this: **`longvt_compression/fast_agent/preflight.py`**.

It encodes the checks below as code. Each check reports the **value it measured**, not just
pass/fail, because the goal is to make you look at numbers rather than at a green light.

**Hard fail vs warning.** 🟧 **OURS** — a check is a hard `FAIL` only when its violation
makes the run's numbers meaningless. Everything else is a `warn`. In particular nothing
asserts a fixed tokens-per-frame: variable-resolution behaviour is *legitimate* (Qwen
resizes by aspect ratio, so tpf differs per video). What is asserted is that the value the
server **produced** matches the value the **same** server **predicted** for the same input.

## Before you run this — predict
You are about to run it against the banked run3. From everything in §5 and §7, **which two
warnings do you expect it to raise?**

In [20]:
cmd = [sys.executable, "-m", "longvt_compression.fast_agent.preflight",
       "--frames", "256", "--expect-retention", "0.25", "--expect-method", "vidcom2",
       "--server-log", f"{OUT}/server_arm_b.log",
       "--proxy-dir", PROXY_DIR, "--proxy-sample", "4",
       "--measure-video", f"{PROXY_DIR}/{ROW['videoID']}_n256.mp4",
       "--out-dir", f"{OUT}/lvbench_c3_notool_seed0", "--expect-resume"]
r = subprocess.run(cmd, capture_output=True, text=True, cwd="/home/cfyang/hanklin",
                   env={**os.environ, "VLLM_LOGGING_LEVEL": "ERROR"})
print(r.stdout)

PREFLIGHT
[  ok  ] python: 3.12.13 (want 3.12 = the `vllm` env)
[  ok  ] interpreter: /local1/cfyang/miniconda3/envs/vllm/bin/python
[  ok  ] vllm: 0.19.0 (expect 0.19.0)
[  ok  ] transformers: 4.57.6
[  ok  ] server launch flags: {'model_tag': '/local1/cfyang/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b', 'enable_auto_tool_choice': True, 'tool_call_parser': 'hermes', 'port': 8035, 'model': '/local1/cfyang/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b', 'allowed_local_media_path': '/local1/cfyang', 'max_model_len': 40960, 'served_model_name': [
[  ok  ] media_io_kwargs num_frames: expect "'num_frames': 256" -- present
[  ok  ] video_pruning_rate: expect "'video_pruning_rate': 0.75" -- present
[ warn ] mm_processor_kwargs max_pixels: NOTE: max_pixels does NOT affect the VIDEO grid in vLLM 0.19 (verified: any value gives the same grid_thw). Only `size.longest_edge` does, and it is a WHOLE-VIDEO budget. Do no

### Interpreting the output

🟩 **RUNTIME** — 20 checks, 0 FAIL, 3 warn. The three warnings are the ones you should have
predicted:

1. `max_pixels does NOT affect the VIDEO grid` — §5.2.
2. `proxy resolution 448x250 -- HARD CEILING` — §3.
3. `surviving tokens/grid-frame: 22.8 … far outside the regime` — §9.

Note the one check that *cannot* be replaced by reading a config file:

```text
[  ok  ] vidcom2 proof-of-life: (EngineCore pid=3877676) [vidcom2_vllm] ACTIVE in pid=3877676
                                 (first mask: thw=(128, 12, 28), q=0.75)
```

That line came from the process that actually computed the mask, and it independently
confirms `T=128` — the same value we derived on CPU in §5.

## The checklist, as a list

Run `preflight.py` before **every** expensive arm. What it covers:

```text
ENVIRONMENT
[ ] python 3.12 and the interpreter path contains envs/vllm   (conda run -n vllm lies)
[ ] vLLM == 0.19.0
[ ] model checkpoint path is the expected snapshot hash

SERVER  (read from the server's own log, not from serve_arm.sh)
[ ] 'num_frames': N present in media_io_kwargs         -- absent = silent resampling
[ ] 'video_pruning_rate': q present, or absent for a control arm
[ ] compressor proof-of-life: [vidcom2_vllm] ACTIVE in pid=  from EngineCore
[ ] /v1/models returns 200 AND a real generation succeeds
[ ] max_pixels: understood to be INERT for video (informational)

PROXIES
[ ] one proxy exists per video at the requested frame count
[ ] proxy duration == source duration (drift < 2%)      -- else every timestamp is wrong
[ ] proxy resolution recorded as the hard ceiling it is

MEASURED, NOT ASSUMED
[ ] frames actually sampled == frames requested
[ ] grid T == raw_frames / 2                            -- temporal_patch_size
[ ] grid_thw and tokens-per-grid-frame printed for this video
[ ] pre-compression total token count printed
[ ] post-compression count measured from <|video_pad|> in the ids
[ ] |actual retention - requested| < 0.01
[ ] surviving tokens/frame printed and compared against ~180 (VidCom2's validated point)

PROMPT AND TOOLS
[ ] --print-prompt run and READ                         -- the 2026-08-13 run1 bug
[ ] prompt style matches --tools (no-tool arm gets the no-tool prompt)
[ ] tool schemas attached iff the prompt mentions tools
[ ] sampling params printed (T, top_p, top_k, penalties, max_tokens)

OUTPUT
[ ] output directory clean, or resume is intended and the skip count is explained
[ ] after the run: sum("error" in r) == 0 BEFORE quoting any accuracy
[ ] at least one trajectory opened and read by a human
```

## The assertions themselves

🟦 **SOURCE** — the hard invariants in `preflight.py`, and why each is hard rather than soft:

| assertion | why hard |
|---|---|
| `actual_frames == requested_frames` | a silently-resampled arm is not the arm you named |
| `grid_T == raw_frames // 2` | if this fails, `temporal_patch_size` changed and every budget is wrong |
| `abs(actual_retention - target) < 0.01` | the compressor did not do what the flag said |
| `mask.sum() == compute_retained_tokens_count(...)` | enforced **inside** `retention.py`; violation corrupts generation |
| `proxy_duration ≈ source_duration` | else every `<t seconds>` marker, and hence every crop, is wrong |
| `"ACTIVE in pid=" in engine_log` | the only proof the compressor you named is the one that ran |
| `sum("error" in row) == 0` | has produced a below-chance accuracy twice in this chapter |

Deliberately **not** asserted (they would break legitimate behaviour):

- a fixed tokens-per-frame — varies by aspect ratio, by design
- a fixed prompt-token count — varies by question length and by video
- equal budgets across arms — §5.2 shows they are **not** equal, and pretending otherwise
  is what hid the problem

### 🔧 Exercise 13
Write a ten-line script yourself that takes a run directory and prints
`n, error_rows, pred_None, accuracy` — and **refuses to print the accuracy** if
`error_rows > 0`. That single habit would have prevented the 17.8% incident.

---
# §14 · Debugging shapes — the cheat sheet

Every shape in Qwen3-VL video inference, with its meaning, for our 256-frame sample.

```text
raw video file            qAIRFyR6NyQ.mp4        1280x720, 5333 s, 159,829 frames
        |  make_skim_proxy.py  (ffmpeg fps=256/5333, scale=448:250)
        v
proxy mp4                 448x250, 256 frames, duration still 5333 s
        |  OpenCVVideoBackend.load_bytes(num_frames=256)
        v
frames array              (256, 250, 448, 3)  uint8
                           ^^^  ^^^  ^^^  ^
                           |    |    |    RGB
                           |    H    W  (source pixels)
                           raw frames
        |  Qwen3VLVideoProcessor: smart_resize + patchify
        v
pixel_values_videos       (46592, 1536)
                           ^^^^^  ^^^^
                           |      1536 = 3 ch x 16 x 16 px x 2 temporal frames
                           |             (one flattened spatio-temporal patch)
                           46592 = T x H x W = 128 x 14 x 26
video_grid_thw            (1, 3) -> [[128, 14, 26]]
                                      ^^^  ^^  ^^
                                      |    |   26 patch cols = 416 px / 16
                                      |    14 patch rows = 224 px / 16
                                      128 temporal steps = 256 raw frames / 2
        |  Qwen3VLVisionModel (27 layers, hidden 1152)  + PatchMerger (2x2 -> 1)
        v
video_embeds              (11648, 4096)
                           ^^^^^  ^^^^
                           |      4096 = out_hidden_size == the LLM's hidden width
                           11648 = T x tpf = 128 x 91
                                   tpf = (14/2) x (26/2) = 7 x 13 = 91
        |  compute_retention_mask(...)  ->  bool (11648,)
        v
mask                      (11648,) bool, exactly 2912 True
        |  emb = emb[mask]
        v
pruned embeds             (2912, 4096)
        |  _create_final_video_embeddings: interleave timestamps + vision markers
        v
final video block         (2912 + 1393, 4096+)     1393 = 128 x ~10.9 structural tokens
        |  + question text, + tool schema block, + prior turns
        v
LLM input sequence        (4305, 4096)   for the skim alone
                          (5,758)        run2 on THIS question, after one crop_video round
                          (<= 40960)     --max-model-len ceiling
```

## The three shape questions that trip people up

**"Why does 256 frames become a temporal grid of 128?"**
`temporal_patch_size = 2`. The ViT's patch embedding consumes **two raw frames as one
temporal patch** — visible in `pixel_values_videos`'s 1536 = 3 × 16 × 16 × **2**. So one
grid step *is* two raw frames, and its `<t seconds>` marker is the **mean** of the pair's
timestamps (🟦 **SOURCE** `qwen3_vl.py:822-826`). Nothing was discarded; two frames were
fused.

**"Why 46,592 patches but only 11,648 tokens?"**
`spatial_merge_size = 2`. The PatchMerger folds each 2×2 block of spatial patches into one
token: 46,592 / 4 = 11,648. The LLM never sees a raw ViT patch.

**"Why is the hidden size 1536 in one place and 4096 in another?"**
1536 is the *input* patch dimension (3 × 16 × 16 × 2 pixels flattened). 1152 is the ViT's
internal width (`vision_config.hidden_size`). 4096 is `out_hidden_size` — what the merger
projects to, equal to the LLM's hidden size so the tokens can be spliced into the text
stream.

### 🔧 Exercise 14
Cover the diagram. Starting from "a 512-frame proxy at 448×250", derive by hand: `grid_thw`,
tokens per grid-frame, total visual tokens, and surviving tokens per frame at r=0.25. Then
verify with `process(...)`. (Careful: `smart_resize`'s budget is whole-video, so tokens per
frame will **not** be 91.)

---
# §15 · What I should now be able to explain

Work through these without the notebook open. If one stalls, the section is named.

## The pipeline
1. Name the sixteen stages from LVBench row to score, and the file that owns each. (§1)
2. Which three stages can be wrong **without raising an exception**? (§1, §12)
3. Why is the skim video modality rather than images? (§1 — because vLLM prunes video only;
   *not* because LongVT does it that way, which is backwards.)

## Geometry
4. Why does a 256-frame input produce `grid_t = 128`? (§5.1, §14)
5. Where exactly does spatial resolution become a visual-token count? (§5.1)
6. Compute the visual tokens for a 448×250 proxy at 64 frames by hand. (§5.1)
7. Why can proxy resolution cap information *before* `max_pixels` is even consulted? (§3, §9)
8. Why does asking for more frames **lower** per-frame resolution? (§5.2)

## VidCom2
9. What tensor does it receive — shape, dtype, device, and what one row means? (§6, §7.4)
10. Why does it save no ViT compute? (§6)
11. How is the per-frame budget computed, and why is it near-uniform in practice? (§7.3)
12. How is token uniqueness computed, and what is it blind to? (§7.2 — the query.)
13. What does `r = 0.25` actually guarantee? (§7.4 — a count, reconciled exactly.)
14. Why does `r = 0.25` **not** imply two configs carry equivalent information? (§9)
15. How do you verify VidCom2 actually executed? (§7.1 — the EngineCore print.)

## EVS
16. What two tensors does the cosine compare, and what does one output scalar mean? (§8)
17. Why is EVS's score temporal-change-based rather than query-aware? (§8)
18. Name two behavioural signatures that reveal EVS ran when the label said VidCom2. (§8 —
    frame 0 fully intact; per-frame variance roughly double.)

## Position and sequence
19. Are mRoPE positions recomputed or sliced after pruning? (§10 — computed on the
    **unpruned** grid, then indexed by the mask.)
20. Why must the mask's `True` count equal the processor's prediction exactly? (§10)

## The experiment
21. What is the localization gap on LVBench, and what does it imply? (§11.2 — +15.62 pp.)
22. How does the model choose its crop start? (§11.1 — it copies a skim timestamp marker.)
23. Give three explanations for run3 losing to run1, and the run that separates them. (§9)
24. Why is `acc | landed` selection-biased? (`RESEARCH.md`, and why run5 exists.)

## Process
25. Name five settings that fail **open** — wrong result, no error. (§12)
26. What do you run before an expensive arm, and what are its hard invariants? (§13)
27. What must you check before quoting any accuracy? (§11.2 — `sum("error" in r)`.)

---

## Open questions this notebook raises but does not settle

🟧 **OURS** — each of these is now a *measurable* question rather than a vague worry:

1. **Was every banked arm at the intended pixel budget?** §5.2 says `max_pixels` never
   applied. The arms all ran at the checkpoint default. Whether that changes any conclusion
   is unknown — but the *recorded* settings and the *actual* settings differ, and
   `RESEARCH.md` should say so.
2. **Is run1 vs run3 budget-matched?** §5.2: 3,584 vs 2,912 tokens on our sample — 19%
   apart, against run3. Measure the distribution across all 103 videos before quoting
   "exactly matched".
3. **Does VidCom2 at its validated operating point beat its own control?** Arm **e**
   (32f hi-res, *no* pruning, 720 tok/frame) is banked at 39.70% over all 1,549. Arm **f**
   (the same skim at r=0.25, 180 tok/frame) is configured but **not banked**. `arm f − arm e`
   is the single most valuable missing number in the chapter: it measures VidCom2's cost at
   the operating point it was validated at, and so separates "compression hurts" from "we
   ran it 8× outside its regime".
4. **Is VidCom2's near-uniform frame budget (§7.3) faithful, or an artefact of
   `SOFTMAX_TEMP = 0.01` in our port?** Check the reference implementation's temperature.
5. **Would a denser skim close the localization gap?** §11.1 shows the agent aims by
   copying skim markers. A 256-frame skim has 41.7 s markers vs 64-frame's 166.7 s.
   run4 (256f + crop) is banked — its hit rate versus run2's is the direct test.

---

## What to change in `RESEARCH.md`

🟧 **OURS** — three corrections this notebook justifies:

1. The "budget | run1/2/5 = 64 frames → 2,688 visual tokens · run3/4 = 256 × 0.25 → 2,688
   (exactly matched, verified)" row is **not verified**. Measured on `qAIRFyR6NyQ`: 3,584 vs
   2,912. Replace with a measured distribution, or state the video it was measured on.
2. Add: **`--mm-processor-kwargs max_pixels` is inert for video in vLLM 0.19.** Arms a–f all
   ran at `size.longest_edge = 25,165,824`. Arms e/f got 720 tok/frame from the hi-res proxy
   plus the default budget, not from the flag.
3. Add to the serving prerequisites: **video `max_pixels` is a whole-video budget**, so
   raising frame count lowers per-frame resolution automatically. This is why 64f gives 112
   tok/frame and 256f gives 91 on the same proxy.